# Conectividad marítima de Chancay y los puertos del Pacífico sudamericano durante 2025–2026

*Maritime connectivity of Chancay and South American Pacific ports during 2025–2026*

**Lizzie Gimena Bartolo Artica · Patricia Goetendia Limache · Bridgit Manani Carrión · Joseph Manuel Orellana Allpoc**  
Universidad Nacional del Centro del Perú · Facultad de Economía  
Huancayo, 19 de octubre de 2026

---

**Análisis reproducible de datos y estimación econométrica**

La investigación examina la evolución de la conectividad marítima de Chancay, Callao y otros doce puertos del Pacífico sudamericano entre 2025T1 y 2026T3. La base principal contiene 98 observaciones de un panel balanceado de catorce puertos y siete trimestres. La variable de estudio es el *Port Liner Shipping Connectivity Index* (PLSCI), publicado por UNCTAD.

La estimación identifica diferencias de nivel y tendencias relativas mediante efectos fijos de puerto y trimestre. Las estadísticas portuarias nacionales constituyen documentación complementaria y se procesan sin mezclarlas con la variable dependiente. El análisis comprende la obtención de fuentes, la depuración, las estimaciones, la evaluación de sensibilidad y el diagnóstico espacial de los residuos.

Las tablas y figuras presentadas corresponden al conjunto de datos archivado para la investigación; se regeneran al ejecutar las celdas respectivas.

## Directorio de trabajo

El procesamiento organiza los archivos originales, las tablas derivadas, la base econométrica, las estimaciones y los gráficos en subdirectorios independientes. La ubicación puede fijarse mediante `RUTA_MANUAL` o seleccionarse en la ventana de diálogo. Los documentos originales se conservan sin modificaciones.

In [ ]:
from pathlib import Path
import os

RUTA_MANUAL = ""  # Directorio del proyecto, si se define una ruta previamente.
ruta = os.environ.get("CHANCAY_DIRECTORIO", "").strip() or RUTA_MANUAL.strip()
if not ruta:
    try:
        import tkinter as tk
        from tkinter import filedialog
        ventana = tk.Tk()
        ventana.withdraw()
        ventana.attributes("-topmost", True)
        ruta = filedialog.askdirectory(title="Seleccionar directorio de resultados")
        ventana.destroy()
    except Exception as exc:
        print("Selector gráfico no disponible:", str(exc)[:160])
if not ruta:
    ruta = input("Directorio de trabajo: ").strip().strip('"')
if not ruta:
    raise ValueError("Es necesario definir el directorio de trabajo.")
CARPETA_TRABAJO = Path(ruta).expanduser().resolve()
CARPETA_TRABAJO.mkdir(parents=True, exist_ok=True)
SUBCARPETAS = {
    "control": CARPETA_TRABAJO / "00_BITACORA_Y_FUENTES",
    "fuentes": CARPETA_TRABAJO / "01_DOCUMENTOS_OFICIALES",
    "tablas": CARPETA_TRABAJO / "02_TABLAS_A_EXCEL",
    "datos": CARPETA_TRABAJO / "03_BASE_ECONOMETRICA",
    "base_final": CARPETA_TRABAJO / "BASE_FINAL",
    "resultados": CARPETA_TRABAJO / "04_ESTIMACIONES",
    "graficos": CARPETA_TRABAJO / "05_GRAFICOS",
    "auditoria": CARPETA_TRABAJO / "06_VERIFICACION",
}
for carpeta in SUBCARPETAS.values():
    carpeta.mkdir(parents=True, exist_ok=True)
print("Directorio de trabajo:", CARPETA_TRABAJO)
for nombre, carpeta in SUBCARPETAS.items():
    print(f"  {nombre:12s} {carpeta}")

## 1. Entorno computacional

La obtención de datos utiliza solicitudes HTTP y consultas a interfaces de programación (API). Los enlaces publicados en páginas institucionales se identifican mediante `BeautifulSoup` y, cuando el contenido se genera dinámicamente, se inspeccionan con Selenium. Los documentos PDF y las hojas de cálculo se procesan con bibliotecas específicas, con reconocimiento óptico de caracteres (OCR) cuando existe el ejecutable correspondiente.

Las estimaciones econométricas se calculan mediante álgebra matricial con NumPy y distribuciones estadísticas de SciPy. La implementación incluye mínimos cuadrados ordinarios (MCO), errores estándar convencionales y la corrección HC3 para heterocedasticidad. Las versiones de los componentes principales quedan identificadas al iniciar el procesamiento.

In [ ]:
import csv
import hashlib
import importlib.util
import io
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import unquote, urljoin, urlparse

DEPENDENCIAS = {"numpy":"numpy", "pandas":"pandas", "scipy":"scipy", "matplotlib":"matplotlib",
                "requests":"requests", "bs4":"beautifulsoup4", "pdfplumber":"pdfplumber",
                "openpyxl":"openpyxl", "pymupdf":"pymupdf",
                "PIL":"Pillow", "pytesseract":"pytesseract"}
faltan = [paquete for modulo, paquete in DEPENDENCIAS.items()
          if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando paquetes faltantes:", ", ".join(faltan))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltan])

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
import pdfplumber
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from types import SimpleNamespace
from IPython.display import display

# Calculamos MCO con matrices y obtenemos los errores estándar convencionales y HC3.
class ModeloMCO:
    def __init__(self, y, x):
        self.y = np.asarray(y, dtype=float).reshape(-1)
        self.x_original = x
        self.X = np.asarray(x, dtype=float)
        if self.X.ndim == 1:
            self.X = self.X.reshape(-1, 1)
        if self.X.shape[0] != self.y.size:
            raise ValueError('El número de filas de X debe coincidir con el de y')

    def fit(self):
        X, y = self.X, self.y
        n, p = X.shape
        beta, _, rango, _ = np.linalg.lstsq(X, y, rcond=None)
        if rango != p or n <= p:
            raise ValueError('La matriz de regresores no tiene rango completo o no hay grados de libertad')
        fitted = X @ beta
        resid = y - fitted
        sse = float(resid @ resid)
        df_resid = n - p
        mse = sse / df_resid
        inv = np.linalg.pinv(X.T @ X, hermitian=True)
        variancias = np.diag(inv) * mse
        bse = np.sqrt(np.maximum(variancias, 0.))
        tvals = beta / bse
        pvals = 2 * stats.t.sf(np.abs(tvals), df_resid)
        H = np.sum((X @ inv) * X, axis=1)
        pesos_hc3 = (resid / np.maximum(1 - H, 1e-12)) ** 2
        meat = X.T @ (X * pesos_hc3[:, None])
        cov_hc3 = inv @ meat @ inv
        robust_bse = np.sqrt(np.maximum(np.diag(cov_hc3), 0.))
        robust_p = 2 * stats.t.sf(np.abs(beta / robust_bse), df_resid)
        centered = np.any(np.all(np.isclose(X, 1.0), axis=0))
        sst = float(np.sum((y - y.mean()) ** 2)) if centered else float(y @ y)
        rsq = 1.0 - sse / sst
        self.rsquared_adj = 1 - (1 - rsq) * (n - 1) / df_resid
        self.rsquared = rsq
        self.aic = n * (np.log(2 * np.pi) + 1 + np.log(sse / n)) + 2 * p
        self.bic = n * (np.log(2 * np.pi) + 1 + np.log(sse / n)) + p * np.log(n)
        self.nobs, self.df_resid, self.mse_resid = int(n), int(df_resid), float(mse)
        self.fittedvalues, self.resid = fitted, resid
        self._robust_bse, self._robust_p = robust_bse, robust_p
        self._ci95 = np.column_stack((beta - stats.t.ppf(.975, df_resid) * bse,
                                      beta + stats.t.ppf(.975, df_resid) * bse))
        if isinstance(self.x_original, pd.DataFrame):
            nombres = self.x_original.columns
            self.params = pd.Series(beta, index=nombres)
            self.bse = pd.Series(bse, index=nombres)
            self.pvalues = pd.Series(pvals, index=nombres)
        else:
            self.params, self.bse, self.pvalues = beta, bse, pvals
        return self

    def conf_int(self):
        return self._ci95

    def get_robustcov_results(self, cov_type='HC3', use_t=True):
        if cov_type != 'HC3':
            raise ValueError('La implementación solo incluye HC3')
        return SimpleNamespace(bse=self._robust_bse, pvalues=self._robust_p)

def agregar_constante(x):
    a = np.asarray(x, dtype=float)
    if a.ndim == 1:
        return np.column_stack((np.ones(len(a)), a))
    return np.column_stack((np.ones(a.shape[0]), a))


print("Versiones cargadas:", "Python", platform.python_version(), "pandas", pd.__version__,
      "NumPy", np.__version__)
prueba = ModeloMCO(np.array([1., 2., 3., 4.]), agregar_constante(np.array([0., 1., 2., 3.]))).fit()
assert np.allclose(prueba.params, [1., 1.], atol=1e-12)
print("Verificación de mínimos cuadrados ordinarios correcta.")

## 2. Cobertura, unidades de observación y fuentes

La unidad de análisis es el par puerto–trimestre. La muestra comprende tres puertos peruanos, ocho chilenos, uno colombiano y dos ecuatorianos, observados en siete trimestres consecutivos entre 2025T1 y 2026T3. El PLSCI de UNCTAD se emplea como medida comparable de conectividad en servicios regulares de contenedores.

Se reúnen, adicionalmente, reportes de la Autoridad Portuaria Nacional del Perú, estadísticas portuarias de Colombia, series del Observatorio Logístico de Chile y el boletín estadístico de Ecuador. Estas fuentes documentan el contexto operativo, con sus unidades y periodicidades originales, y **no se incorporan como regresores** del modelo principal. Las consultas automatizadas respetan los límites de acceso publicados por cada servicio.

In [ ]:
# Parámetros de descarga y códigos de los puertos incluidos en la muestra.
ACTIVAR_SELENIUM = os.environ.get("PACIFICO_SELENIUM", "1") == "1"
SELENIUM_WORKERS = 3
WORKERS_DESCARGA = 8
REFRESCAR_DOCUMENTOS = False   # En una repetición se reutilizan archivos descargados y verificados.
TIMEOUT_HTTP = (8, 45)        # Conexión y tiempo máximo de espera por respuesta.
MAX_APN_DOCUMENTOS = 36
WORKERS_CHILE = 1            # Consultas consecutivas para respetar el límite de la API.
REGISTROS_POR_PAGINA_CHILE = 100  # Registros máximos por página.
WORKERS_PDF = 2              # Procesamiento de documentos en paralelo.
USAR_OCR_PDF = os.environ.get("PACIFICO_OCR", "1") == "1"
PAGINAS_PDF_A_EXAMINAR = int(os.environ.get("PACIFICO_LIMITE_PDF", "0"))  # Cero permite revisar todo el documento.
# El límite de consultas a Chile se aplica independientemente del panel PLSCI.
EXIGIR_CHILE_COMPLETO = False
INTERVALO_CHILE_SEG = 2.5          # Pausa mínima entre peticiones que sí llegan a internet.
REINTENTOS_CHILE = 5               # Número máximo de intentos por página pendiente.
ESPERA_CHILE_429_INICIAL = 30.0    # Espera inicial cuando la API limita las consultas.
ESPERA_CHILE_MAX_SEG = 180.0        # Tope para las esperas calculadas localmente.
PRESUPUESTO_CHILE_SEG = 540.0      # Tiempo máximo para consultar páginas pendientes en esta ejecución.

N_PERMUTACIONES = 9999
N_PERMUTACIONES_TRIMESTRE = 1999
SEMILLA = 20261014

PERIODOS = ['2025T1','2025T2','2025T3','2025T4','2026T1','2026T2','2026T3']
CODIGOS = ['PECHY','PECLL','PEPAI','CLANF','CLARI','CLCNL','CLIQQ',
           'CLPAG','CLSAI','CLSVE','CLVAP','COBUN','ECGYE','ECPBO']
NOMBRES = ['Chancay','Callao','Paita','Antofagasta','Arica','Coronel','Iquique',
           'Puerto Angamos','San Antonio','San Vicente','Valparaíso',
           'Buenaventura','Guayaquil','Puerto Bolívar']
NOMBRE_PUERTO = dict(zip(CODIGOS, NOMBRES))
CUATRO_REFERENTES = ['CLSAI','CLVAP','COBUN','ECGYE']

FUENTES_URL = {
    'unctad_plsci': os.environ.get('CHANCAY_URL_PLSCI',
        'https://unctadstat-api.unctad.org/bulkdownload/US.PLSCI/US_PLSCI?fileType=csv'),
    'unctad_lsbci': os.environ.get('CHANCAY_URL_LSBCI',
        'https://unctadstat-api.unctad.org/bulkdownload/US.LSBCI/US_LSBCI?fileType=csv'),
    'apn_portal': os.environ.get('CHANCAY_URL_APN',
        'https://www.gob.pe/institucion/apn/informes-publicaciones/6573509-estadisticas-apn-2025-trafico-de-carga'),
    'colombia_socrata': os.environ.get('CHANCAY_URL_COLOMBIA',
        'https://www.datos.gov.co/resource/5r3g-zv5z.csv?$limit=50000'),
    'colombia_portal': os.environ.get('PACIFICO_URL_COLOMBIA_PORTAL',
        'https://www.supertransporte.gov.co/index.php/superintendencia-delegada-de-puertos/estadisticas-trafico-portuario-en-colombia/'),
    'chile_portal': os.environ.get('PACIFICO_URL_CHILE_PORTAL',
        'https://observatorio.observatoriologistico.cl/datos/contenedores-transferidos-en-terminales-estatales-mensualizado'),
    'chile_api': os.environ.get('CHANCAY_URL_CHILE_API',
        'https://www.observatoriologistico.cl/api/v1/datastreams'),
    'ecuador_portal': os.environ.get('PACIFICO_URL_ECUADOR_PORTAL',
        'https://www.mit.gob.ec/boletines-estadisticos-portuarios/'),
    'ecuador_pdf': os.environ.get('CHANCAY_URL_ECUADOR',
        'https://www.camae.org/wp-content/uploads/2026/08/Estadisitica-Porturaria-2025.pdf?x35628='),
}
PAGINAS_WEB = {k:FUENTES_URL[k] for k in
               ['apn_portal','colombia_portal','chile_portal','ecuador_portal']}
FUENTES_APN_ENLACES_DOCUMENTADOS = [
    {
        "clave": "APN2025_01_pdf",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/7783523/6573509-movimiento-de-contenedores-e"
            "n-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-enero-2025.pdf?v=1742222410"
        ),
        "formato": "pdf",
    },
    {
        "clave": "APN2025_03_xlsx",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/7783525/6573509-resumen-del-movimiento-de-ca"
            "rga-en-los-terminales-portuarios-de-uso-publico-enero-2025.xlsx?v=1742222412"
        ),
        "formato": "xlsx",
    },
    {
        "clave": "APN2025_04_pdf",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/8133807/6573509-movimiento-de-contenedores-e"
            "n-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-febrero-2025.pdf?v=1748449938"
        ),
        "formato": "pdf",
    },
    {
        "clave": "APN2025_06_xlsx",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/8133809/6573509-resumen-del-movimiento-de-ca"
            "rga-en-los-terminales-portuarios-de-uso-publico-febrero-2025.xlsx?v=1748449939"
        ),
        "formato": "xlsx",
    },
    {
        "clave": "APN2025_07_pdf",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/8133810/6573509-movimiento-de-contenedores-e"
            "n-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-marzo-2025.pdf?v=1748449940"
        ),
        "formato": "pdf",
    },
    {
        "clave": "APN2025_09_xlsx",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/8133812/6573509-resumen-del-movimiento-de-ca"
            "rga-en-los-terminales-portuarios-de-uso-publico-marzo-2025.xlsx?v=1748449942"
        ),
        "formato": "xlsx",
    },
    {
        "clave": "APN2025_10_pdf",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/8188129/6573509-movimiento-de-contenedores-e"
            "n-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-abril-2025.pdf?v=1749491317"
        ),
        "formato": "pdf",
    },
    {
        "clave": "APN2025_12_xlsx",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/8188131/6573509-resumen-del-movimiento-de-ca"
            "rga-en-los-terminales-portuarios-de-uso-publico-abril-2025.xlsx?v=1749491319"
        ),
        "formato": "xlsx",
    },
    {
        "clave": "APN2025_13_pdf",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/8324141/6573509-movimiento-de-contenedores-e"
            "n-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-mayo-2025.pdf?v=1751893897"
        ),
        "formato": "pdf",
    },
    {
        "clave": "APN2025_15_xlsx",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/8324143/6573509-resumen-del-movimiento-de-ca"
            "rga-en-los-terminales-portuarios-de-uso-publico-mayo-2025.xlsx?v=1751893898"
        ),
        "formato": "xlsx",
    },
    {
        "clave": "APN2025_16_pdf",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/8462381/6573509-movimiento-de-contenedores-e"
            "n-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-junio-2025.pdf?v=1754681188"
        ),
        "formato": "pdf",
    },
    {
        "clave": "APN2025_18_xlsx",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/8462383/6573509-resumen-del-movimiento-de-ca"
            "rga-en-los-terminales-portuarios-de-uso-publico-junio-2025.xlsx?v=1754681189"
        ),
        "formato": "xlsx",
    },
    {
        "clave": "APN2025_19_pdf",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/8592710/6573509-movimiento-de-contenedores-e"
            "n-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-julio-2025.pdf?v=1756910786"
        ),
        "formato": "pdf",
    },
    {
        "clave": "APN2025_21_xlsx",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/8592712/6573509-resumen-del-movimiento-de-ca"
            "rga-en-los-terminales-portuarios-de-uso-publico-julio-2025.xlsx?v=1756910787"
        ),
        "formato": "xlsx",
    },
    {
        "clave": "APN2025_22_pdf",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/8843486/6573509-movimiento-de-contenedores-e"
            "n-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-agosto-2025.pdf?v=1760712257"
        ),
        "formato": "pdf",
    },
    {
        "clave": "APN2025_24_xlsx",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/8843488/6573509-resumen-del-movimiento-de-ca"
            "rga-en-los-terminales-portuarios-de-uso-publico-agosto-2025.xlsx?v=1760712258"
        ),
        "formato": "xlsx",
    },
    {
        "clave": "APN2025_25_pdf",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/8957096/6573509-movimiento-de-contenedores-e"
            "n-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-septiembre-2025.pdf?v=1762466"
            "071"
        ),
        "formato": "pdf",
    },
    {
        "clave": "APN2025_27_xlsx",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/8957098/6573509-resumen-del-movimiento-de-ca"
            "rga-en-los-terminales-portuarios-de-uso-publico-septiembre-2025.xlsx?v=1762466072"
        ),
        "formato": "xlsx",
    },
    {
        "clave": "APN2025_28_pdf",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/9089690/6573509-movimiento-de-contenedores-e"
            "n-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-octubre-2025.pdf?v=1764707447"
        ),
        "formato": "pdf",
    },
    {
        "clave": "APN2025_30_xlsx",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/9089692/6573509-resumen-del-movimiento-de-ca"
            "rga-en-los-terminales-portuarios-de-uso-publico-octubre-2025.xlsx?v=1764707449"
        ),
        "formato": "xlsx",
    },
    {
        "clave": "APN2025_31_pdf",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/9260719/6573509-movimiento-de-contenedores-e"
            "n-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-noviembre-2025.pdf?v=17678813"
            "40"
        ),
        "formato": "pdf",
    },
    {
        "clave": "APN2025_33_xlsx",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/9260721/6573509-resumen-del-movimiento-de-ca"
            "rga-en-los-terminales-portuarios-de-uso-publico-noviembre-2025.xlsx?v=1767881341"
        ),
        "formato": "xlsx",
    },
    {
        "clave": "APN2025_34_pdf",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/9409816/6573509-movimiento-de-contenedores-e"
            "n-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-diciembre-2025.pdf?v=17704125"
            "54"
        ),
        "formato": "pdf",
    },
    {
        "clave": "APN2025_36_xlsx",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/9409818/6573509-resumen-del-movimiento-de-ca"
            "rga-en-los-terminales-portuarios-de-uso-publico-diciembre-2025.xlsx?v=1770412556"
        ),
        "formato": "xlsx",
    },
    {
        "clave": "APN2025_37_pdf",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/9563803/6573509-movimiento-de-contenedores-e"
            "n-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-anual-2025.pdf?v=1772817243"
        ),
        "formato": "pdf",
    },
    {
        "clave": "APN2025_40_pdf",
        "url": (
            "https://cdn.www.gob.pe/uploads/document/file/9563806/6573509-resumen-del-movimiento-de-ca"
            "rga-en-los-terminales-portuarios-de-uso-publico-anual-2025.pdf?v=1772817243"
        ),
        "formato": "pdf",
    },
]

print("Puertos:", len(CODIGOS), "| Trimestres:", len(PERIODOS),
      "| Selenium:", "activo" if ACTIVAR_SELENIUM else "inactivo",
      "| Sesiones configuradas:", SELENIUM_WORKERS,
      "| Descargas HTTP en paralelo:", WORKERS_DESCARGA)
print("Fuente PLSCI:", FUENTES_URL['unctad_plsci'])

## 3. Obtención de los índices de UNCTAD

Se descargan los archivos oficiales del *Port Liner Shipping Connectivity Index* (US.PLSCI) y del *Liner Shipping Bilateral Connectivity Index* (US.LSBCI). El primero proporciona las observaciones del panel econométrico; el segundo se utiliza exclusivamente para construir ponderaciones exploratorias basadas en conectividad bilateral entre países.

Los archivos fuente se conservan en su formato de publicación. Para cada descarga se registra su procedencia, estado HTTP, tamaño y huella criptográfica SHA-256. La disponibilidad de US.PLSCI es necesaria para estimar el modelo principal.

In [ ]:
RUTA_DESCARGAS_ANTERIORES=SUBCARPETAS['control']/'01_DESCARGAS_HTTP.csv'
try:
    DESCARGAS_PREVIAS=pd.read_csv(RUTA_DESCARGAS_ANTERIORES).fillna('').to_dict('records')
except (OSError,ValueError):
    DESCARGAS_PREVIAS=[]
REGISTRO_DESCARGAS = []

def sesion_http():
    s = requests.Session()
    politica = Retry(total=2, connect=2, read=1, status=2,
                     backoff_factor=0.8,
                     status_forcelist=[429, 500, 502, 503, 504],
                     allowed_methods=['GET'], respect_retry_after_header=True)
    s.mount('https://', HTTPAdapter(max_retries=politica, pool_connections=6, pool_maxsize=6))
    s.mount('http://', HTTPAdapter(max_retries=politica, pool_connections=6, pool_maxsize=6))
    s.headers.update({'User-Agent': 'Mozilla/5.0 (compatible; UNCP-Estudio-Portuario/1.0)',
                      'Accept': '*/*'})
    return s

def sha256_archivo(archivo):
    h = hashlib.sha256()
    with open(archivo, 'rb') as f:
        for bloque in iter(lambda: f.read(1024 * 1024), b''):
            h.update(bloque)
    return h.hexdigest()

def validar_tipo(archivo, extension):
    with open(archivo, 'rb') as f:
        inicio = f.read(2048).lstrip(b'\xef\xbb\xbf')
    ext = extension.lower().lstrip('.')
    if ext == 'pdf': return inicio.startswith(b'%PDF')
    if ext in ('xlsx','xlsm'): return inicio.startswith(b'PK')
    if ext == '7z': return inicio.startswith(b'7z\xbc\xaf\x27\x1c') or b',' in inicio[:256]
    if ext == 'json':
        try: json.loads(Path(archivo).read_text(encoding='utf-8-sig')); return True
        except (ValueError, UnicodeError): return False
    if ext == 'csv':
        return (b',' in inicio or b';' in inicio) and not inicio.lower().startswith(b'<html')
    return len(inicio)>0

def descargar_archivo(clave, url, destino, tipo, obligatorio=False):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    registro = {'clave':clave,'url_solicitada':url,'archivo':str(destino),
                'fecha_utc':datetime.now(timezone.utc).isoformat(),
                'tipo_esperado':tipo, 'obligatorio':obligatorio}
    if destino.exists() and not REFRESCAR_DOCUMENTOS and validar_tipo(destino,tipo):
        registro.update(estado='EXISTENTE_VALIDADO',http=200,bytes=destino.stat().st_size,
                        sha256=sha256_archivo(destino),url_final=url)
        return registro
    parcial=destino.with_name(destino.name+'.part')
    try:
        with sesion_http() as sesion:
            with sesion.get(url, stream=True, timeout=TIMEOUT_HTTP) as respuesta:
                registro.update(http=respuesta.status_code,url_final=respuesta.url)
                respuesta.raise_for_status()
                longitud = 0
                with parcial.open('wb') as f:
                    for bloque in respuesta.iter_content(chunk_size=262144):
                        if bloque:
                            longitud+=len(bloque);f.write(bloque)
                if longitud == 0 or not validar_tipo(parcial,tipo):
                    raise ValueError(f'Formato {tipo} inválido, documento vacío o página de error en lugar del archivo')
                parcial.replace(destino)
                registro.update(estado='DESCARGADO_VALIDADO',bytes=longitud,
                                sha256=sha256_archivo(destino))
    except Exception as exc:
        if parcial.exists(): parcial.unlink()
        registro.update(estado='ERROR', error=f'{type(exc).__name__}: {str(exc)[:280]}')
    return registro

# El archivo masivo de UNCTAD puede entregarse comprimido en formato 7z.
TAREAS_UNCTAD = [
    ('UNCTAD_PLSCI', FUENTES_URL['unctad_plsci'],
     SUBCARPETAS['fuentes'] / 'UNCTAD' / 'US_PLSCI_descarga_original.7z','7z',True),
    ('UNCTAD_LSBCI', FUENTES_URL['unctad_lsbci'],
     SUBCARPETAS['fuentes'] / 'UNCTAD' / 'US_LSBCI_descarga_original.7z','7z',False),
]
with ThreadPoolExecutor(max_workers=2) as pool:
    futuros = {pool.submit(descargar_archivo,*t):t[0] for t in TAREAS_UNCTAD}
    for futuro in as_completed(futuros):
        resultado=futuro.result()
        REGISTRO_DESCARGAS.append(resultado)
        print(resultado['clave'], resultado['estado'], resultado.get('bytes',''),
              resultado.get('error','')[:180])

if not any(r['clave']=='UNCTAD_PLSCI' and r['estado']!='ERROR' for r in REGISTRO_DESCARGAS):
    pd.DataFrame(REGISTRO_DESCARGAS).to_csv(
        SUBCARPETAS['control']/'01_DESCARGAS_HTTP.csv',index=False,encoding='utf-8-sig')
    raise RuntimeError('No se pudo descargar el PLSCI desde la API de UNCTAD. Revisar 01_DESCARGAS_HTTP.csv.')

### 3.1. Extracción y validación de los archivos de origen

Una vez recibidos los archivos de UNCTAD, se extraen los CSV y se comprueba su estructura, la identificación de los puertos y la codificación temporal. Las observaciones se seleccionan después de verificar las columnas originales, sin efectuar imputaciones ni sustituir ausencias por ceros.

In [ ]:
def extraer_unctad(archivo_fuente, destino, nombre_esperado):
    destino=Path(destino);destino.mkdir(parents=True, exist_ok=True)
    contenido=Path(archivo_fuente).read_bytes()[:8]
    if contenido.startswith(b'Quarter,') or contenido.startswith(b'\xef\xbb\xbfQuarter,'):
        salida=destino/nombre_esperado
        shutil.copy2(archivo_fuente,salida)
        return salida
    if not contenido.startswith(b"7z\xbc\xaf\x27\x1c"):
        raise ValueError('El servidor no ha devuelto un contenedor 7z ni un CSV reconocible')
    try:
        if importlib.util.find_spec('py7zr') is None:
            from ctypes.util import find_library
            if find_library('archive'):
                raise ImportError('Usando libarchive para esta plataforma')
            subprocess.check_call([sys.executable,'-m','pip','install','py7zr'])
        import py7zr
        with py7zr.SevenZipFile(archivo_fuente,mode='r') as z:
            archivos=z.getnames()
            for nombre in archivos:
                if Path(nombre).is_absolute() or '..' in Path(nombre).parts:
                    raise ValueError('Ruta insegura en el archivo comprimido')
            z.extractall(path=destino)
    except Exception as exc_py7zr:
        # Otra forma de extraer 7z cuando está disponible libarchive.
        import ctypes as ct
        from ctypes.util import find_library
        ubicacion = find_library('archive')
        if not ubicacion:
            raise RuntimeError('No fue posible abrir 7z. Instala py7zr en el kernel.') from exc_py7zr
        lib=ct.CDLL(ubicacion)
        lib.archive_read_new.restype=ct.c_void_p
        lib.archive_read_support_filter_all.argtypes=[ct.c_void_p]
        lib.archive_read_support_format_all.argtypes=[ct.c_void_p]
        lib.archive_read_open_filename.argtypes=[ct.c_void_p,ct.c_char_p,ct.c_size_t]
        lib.archive_read_next_header.argtypes=[ct.c_void_p,ct.POINTER(ct.c_void_p)]
        lib.archive_entry_pathname.argtypes=[ct.c_void_p]
        lib.archive_entry_pathname.restype=ct.c_char_p
        lib.archive_read_data.argtypes=[ct.c_void_p,ct.c_void_p,ct.c_size_t]
        lib.archive_read_data.restype=ct.c_ssize_t
        lib.archive_read_free.argtypes=[ct.c_void_p]
        handle=lib.archive_read_new()
        try:
            lib.archive_read_support_filter_all(handle)
            lib.archive_read_support_format_all(handle)
            if lib.archive_read_open_filename(handle,str(archivo_fuente).encode('utf8'),8192)!=0:
                raise ValueError('No fue posible abrir el archivo con libarchive')
            entrada=ct.c_void_p()
            while lib.archive_read_next_header(handle,ct.byref(entrada))==0:
                nombre=lib.archive_entry_pathname(entrada).decode('utf8')
                if Path(nombre).is_absolute() or '..' in Path(nombre).parts:
                    raise ValueError('Ruta insegura en el contenedor')
                ruta=destino/nombre
                ruta.parent.mkdir(parents=True,exist_ok=True)
                with ruta.open('wb') as f:
                    buf=ct.create_string_buffer(262144)
                    while True:
                        leidos=lib.archive_read_data(handle,buf,len(buf))
                        if leidos<0: raise ValueError('Fallo de extracción 7z')
                        if leidos==0: break
                        f.write(buf.raw[:leidos])
        finally:
            lib.archive_read_free(handle)
    candidatos=list(destino.rglob('*.csv'))
    if not candidatos: raise FileNotFoundError('El archivo oficial no contenía un CSV')
    por_nombre=[p for p in candidatos if nombre_esperado.lower() in p.name.lower()]
    return por_nombre[0] if por_nombre else candidatos[0]

PLSCI_ORIGEN = extraer_unctad(TAREAS_UNCTAD[0][2],
    SUBCARPETAS['fuentes']/'UNCTAD'/'PLSCI_CSV', 'US_PLSCI.csv')
print('PLSCI extraído:', PLSCI_ORIGEN.name, '| bytes:',PLSCI_ORIGEN.stat().st_size)
LSBCI_ORIGEN = None
if TAREAS_UNCTAD[1][2].exists():
    try:
        LSBCI_ORIGEN = extraer_unctad(TAREAS_UNCTAD[1][2],
            SUBCARPETAS['fuentes']/'UNCTAD'/'LSBCI_CSV','US_LSBCI.csv')
        print('LSBCI extraído:',LSBCI_ORIGEN.name)
    except Exception as exc:
        print('No se pudo procesar LSBCI:',str(exc)[:250])

def leer_csv_unctad(archivo, columnas):
    # Conservamos los códigos de puerto como texto.
    tabla=pd.read_csv(archivo,low_memory=False,dtype='string')
    if not set(columnas).issubset(tabla.columns):
        raise ValueError(f'Estructura inesperada en {archivo.name}: {tabla.columns.tolist()[:10]}')
    return tabla

primera_lectura=pd.read_csv(PLSCI_ORIGEN,nrows=3)
print('Columnas oficiales PLSCI:',primera_lectura.columns.tolist())

## 4. Documentación estadística portuaria nacional

Se recopilan las publicaciones oficiales de Perú, Chile, Colombia y Ecuador para conservar un respaldo documental de la actividad portuaria regional. La inspección de portales combina peticiones HTTP con Selenium cuando la página presenta contenido dinámico. Las descargas y sus respectivas direcciones de origen se registran en una bitácora, y los archivos originales permanecen separados de las tablas extraídas.

In [ ]:
def enlaces_documentales(html, pagina):
    sopa=BeautifulSoup(html,'html.parser')
    encontrados=[]
    for a in sopa.find_all('a',href=True):
        u=urljoin(pagina,a['href'].strip())
        texto=a.get_text(' ',strip=True)[:170]
        if urlparse(u).scheme not in ('http','https'): continue
        if re.search(r'\.(?:pdf|xlsx|xls|csv|7z)(?:\?|$)',u,re.I) or 'cdn.www.gob.pe/uploads/document' in u:
            encontrados.append({'pagina':pagina,'texto':texto,'url_documento':u})
    return encontrados

def html_requests(url):
    with sesion_http() as s:
        r=s.get(url,timeout=TIMEOUT_HTTP)
        r.raise_for_status()
        return r.text

def revisar_portal_selenium(nombre,url):
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    options=webdriver.ChromeOptions()
    options.add_argument('--headless=new')
    options.add_argument('--disable-gpu')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--no-sandbox')
    options.add_argument('--window-size=1440,1050')
    options.page_load_strategy='eager'
    navegador=webdriver.Chrome(options=options)
    try:
        navegador.set_page_load_timeout(22)
        navegador.get(url)
        WebDriverWait(navegador,12).until(EC.presence_of_element_located((By.TAG_NAME,'body')))
        for y in [0.45,1.0]:
            navegador.execute_script('window.scrollTo(0,document.body.scrollHeight*arguments[0]);',y)
            time.sleep(.6)
        return navegador.page_source
    finally:
        navegador.quit()

html_entradas={}
observaciones_portales=[]
for nombre,url in PAGINAS_WEB.items():
    try:
        html=html_requests(url)
        html_entradas[nombre]=html
        observaciones_portales.append({'portal':nombre,'metodo':'requests','estado':'OK',
                                       'enlaces':len(enlaces_documentales(html,url))})
        (SUBCARPETAS['control']/f'PORTAL_{nombre}_HTTP.html').write_text(html,encoding='utf8')
    except Exception as exc:
        observaciones_portales.append({'portal':nombre,'metodo':'requests',
                                       'estado':'ERROR','detalle':str(exc)[:230]})

if ACTIVAR_SELENIUM:
    if importlib.util.find_spec('selenium') is None:
        try:
            subprocess.check_call([sys.executable,'-m','pip','install','selenium'])
        except Exception as exc:
            print('No pudo instalarse Selenium; se conservan los resultados HTTP:',str(exc)[:180])
    if importlib.util.find_spec('selenium') is not None:
        print('Consultando portales con',SELENIUM_WORKERS,'navegadores Selenium en paralelo...')
        with ThreadPoolExecutor(max_workers=SELENIUM_WORKERS) as pool:
            futuros={pool.submit(revisar_portal_selenium,nombre,url):nombre
                     for nombre,url in PAGINAS_WEB.items()}
            for futuro in as_completed(futuros):
                nombre=futuros[futuro]
                try:
                    html=futuro.result()
                    html_entradas[nombre]=html
                    urls=enlaces_documentales(html,PAGINAS_WEB[nombre])
                    (SUBCARPETAS['control']/f'PORTAL_{nombre}_SELENIUM.html').write_text(html,encoding='utf8')
                    observaciones_portales.append({'portal':nombre,'metodo':'selenium',
                                                   'estado':'OK','enlaces':len(urls)})
                    print('Selenium',nombre,'enlaces:',len(urls))
                except Exception as exc:
                    observaciones_portales.append({'portal':nombre,'metodo':'selenium',
                                                   'estado':'ERROR','detalle':str(exc)[:230]})
                    print('Selenium',nombre,'no disponible:',str(exc)[:150])

pd.DataFrame(observaciones_portales).to_csv(
    SUBCARPETAS['control']/'02_INSPECCION_PORTALES.csv',index=False,encoding='utf-8-sig')

enlaces_todos=[]
for nombre,html in html_entradas.items():
    enlaces_todos.extend(enlaces_documentales(html,PAGINAS_WEB[nombre]))
if enlaces_todos:
    pd.DataFrame(enlaces_todos).drop_duplicates('url_documento').to_csv(
        SUBCARPETAS['control']/'03_ENLACES_DESCUBIERTOS.csv',index=False,encoding='utf-8-sig')
print('Enlaces documentales descubiertos:',len({x['url_documento'] for x in enlaces_todos}))

### 4.1. Publicaciones de Perú, Colombia y Ecuador

En Perú se recopilan reportes mensuales de movimiento de carga y contenedores de la Autoridad Portuaria Nacional (APN). Para Colombia se consulta el conjunto de tráfico portuario publicado mediante Socrata. En Ecuador se conserva el boletín estadístico portuario de 2025, del que se obtiene una tabla específica para la revisión documental.

Los reportes se almacenan con su formato original y se generan archivos tabulares derivados cuando la información lo permite. Esta documentación no altera la construcción del panel PLSCI.

In [ ]:
def nombre_seguro(s):
    s=unquote(str(s))
    s=re.sub(r'[^A-Za-z0-9._-]+','_',s).strip('._')
    return s[:105] or 'documento'

# Incorporamos los enlaces a boletines que publica la APN.
trabajos=[]
lista_apn=[(x['clave'],x['url'],x['formato']) for x in FUENTES_APN_ENLACES_DOCUMENTADOS]
for registro in enlaces_todos:
    u=registro['url_documento']
    if ('cdn.www.gob.pe' in urlparse(u).netloc or 'gob.pe' in urlparse(u).netloc) and \
       re.search(r'\.(pdf|xlsx)(?:\?|$)',u,re.I) and '2025' in u.lower():
        ext=re.search(r'\.(pdf|xlsx)(?:\?|$)',u,re.I).group(1).lower()
        lista_apn.append(('APN_PORTAL_'+str(len(lista_apn)+1),u,ext))
seen=set()
for clave,u,ext in lista_apn:
    canon=u.split('?')[0]
    if canon in seen: continue
    seen.add(canon)
    ruta=SUBCARPETAS['fuentes']/'PERU_APN'/f'{len(seen):02d}_{nombre_seguro(Path(urlparse(u).path).name)}'
    trabajos.append((clave,u,ruta,ext,False))
    if len(seen)>=MAX_APN_DOCUMENTOS:break

# Descargamos los registros colombianos en CSV.
trabajos.append(('COLOMBIA_SOCRATA',FUENTES_URL['colombia_socrata'],
                 SUBCARPETAS['fuentes']/'COLOMBIA'/'TRAFICO_PORTUARIO_SOCRATA.csv','csv',False))
trabajos.append(('ECUADOR_BOLETIN_2025',FUENTES_URL['ecuador_pdf'],
                 SUBCARPETAS['fuentes']/'ECUADOR'/'BOLETIN_PORTUARIO_2025.pdf','pdf',False))

# Guardamos los boletines PDF publicados por Colombia.
colombia_pdf=[d['url_documento'] for d in enlaces_todos
              if 'supertransporte.gov.co' in urlparse(d['url_documento']).netloc and \
                 '.pdf' in urlparse(d['url_documento']).path.lower()]
for idx,u in enumerate(dict.fromkeys(colombia_pdf)):
    if idx>=3:break
    trabajos.append((f'COLOMBIA_BOLETIN_{idx+1}',u,
                     SUBCARPETAS['fuentes']/'COLOMBIA'/f'BOLETIN_{idx+1:02d}.pdf','pdf',False))

print('Descargando en paralelo',len(trabajos),'documentos oficiales...')
with ThreadPoolExecutor(max_workers=WORKERS_DESCARGA) as pool:
    futuros={pool.submit(descargar_archivo,*t):t[0] for t in trabajos}
    for futuro in as_completed(futuros):
        resultado=futuro.result()
        REGISTRO_DESCARGAS.append(resultado)
        print(resultado['clave'],resultado['estado'],
              str(resultado.get('error',''))[:100])

pd.DataFrame(REGISTRO_DESCARGAS).to_csv(
    SUBCARPETAS['control']/'01_DESCARGAS_HTTP.csv',index=False,encoding='utf-8-sig')
print('Descargas completas y verificadas:',sum(r['estado']!='ERROR' for r in REGISTRO_DESCARGAS),
      'de',len(REGISTRO_DESCARGAS))

### 4.2. Series del Observatorio Logístico de Chile

Se consideran las series P048, relativa al movimiento de contenedores, y P012, referente a carga transferida en toneladas. La API se consulta por páginas con controles de integridad: número de registros, identificación del conjunto y correspondencia con los metadatos de paginación.

Las páginas válidas se conservan para posibilitar la reanudación de consultas interrumpidas. Las respuestas HTTP 429 se gestionan mediante intervalos de espera y reintentos limitados. Solo se consolida un conjunto cuando coincide el total de registros efectivamente recuperados con el declarado por la fuente. Estas series permanecen separadas del panel trimestral de UNCTAD.

In [ ]:
# Las páginas chilenas se guardan por separado y se comprueban antes de reutilizarlas.
from email.utils import parsedate_to_datetime
import random

REGISTRO_CHILE_PAGINAS = []
_RUTA_INTENTOS_CHILE = SUBCARPETAS['control'] / '04B_CHILE_REINTENTOS_HTTP.csv'
try:
    REGISTRO_CHILE_INTENTOS = pd.read_csv(_RUTA_INTENTOS_CHILE).fillna('').to_dict('records')
except (FileNotFoundError, ValueError, pd.errors.EmptyDataError):
    REGISTRO_CHILE_INTENTOS = []
_ULTIMA_CONSULTA_CHILE = [0.0]
_INICIO_DESCARGA_CHILE = time.monotonic()

def registrar_intento_chile(codigo, pagina, intento, http, estado, espera, detalle):
    registro = {
        'codigo': codigo, 'pagina': int(pagina), 'intento': int(intento),
        'fecha_utc': datetime.now(timezone.utc).isoformat(),
        'http': http, 'estado': estado,
        'espera_s': round(float(espera), 2), 'detalle': str(detalle)[:220],
    }
    REGISTRO_CHILE_INTENTOS.append(registro)
    # Guardamos los intentos conforme avanza la descarga.
    pd.DataFrame(REGISTRO_CHILE_INTENTOS).to_csv(
        _RUTA_INTENTOS_CHILE, index=False, encoding='utf-8-sig')

def espaciar_consultas_chile():
    ahora = time.monotonic()
    transcurrido = ahora - _ULTIMA_CONSULTA_CHILE[0]
    faltante = INTERVALO_CHILE_SEG - transcurrido
    if faltante > 0:
        time.sleep(faltante)
    _ULTIMA_CONSULTA_CHILE[0] = time.monotonic()

def segundos_retry_after(valor):
    """Interpreta Retry-After como segundos o como fecha HTTP."""
    if not valor:
        return None
    try:
        numero = float(str(valor).strip())
        return max(0.0, numero)
    except (TypeError, ValueError):
        pass
    try:
        fecha = parsedate_to_datetime(str(valor))
        if fecha.tzinfo is None:
            fecha = fecha.replace(tzinfo=timezone.utc)
        return max(0.0, (fecha - datetime.now(timezone.utc)).total_seconds())
    except (TypeError, ValueError, OverflowError):
        return None

def pausa_entre_reintentos_chile(intento, respuesta=None, por_tasa=False):
    # Se espacian las consultas para respetar el límite de la API.
    indicado = segundos_retry_after(respuesta.headers.get('Retry-After')) if respuesta is not None else None
    progresivo = (ESPERA_CHILE_429_INICIAL if por_tasa else 3.0) * (2 ** max(0, intento-1))
    segundos = indicado if indicado is not None else min(ESPERA_CHILE_MAX_SEG, progresivo)
    return max(0.5, float(segundos))

def obtener_pagina_chile(codigo, pagina, esperado=None, total_esperado=None):
    carpeta = SUBCARPETAS['fuentes'] / 'CHILE'
    carpeta.mkdir(parents=True, exist_ok=True)
    destino = carpeta / f'{codigo}_2025_PAGINA_{pagina:03d}.json'
    url = FUENTES_URL['chile_api'].rstrip('/') + f'/{codigo}/data'
    params = {'page': pagina, 'limit': REGISTROS_POR_PAGINA_CHILE, 'anio': 2025}

    def validar_json(contenido):
        if not isinstance(contenido, dict) or contenido.get('status') != 'success':
            raise ValueError('La API no respondió status=success')
        registros = contenido.get('data')
        if not isinstance(registros, list):
            raise ValueError('La respuesta no contiene una lista de registros')
        meta = contenido.get('pagination')
        if not isinstance(meta, dict):
            raise ValueError('Faltan los metadatos de paginación')
        try:
            n_pagina = int(meta['page'])
            limite = int(meta['limit'])
            n_total = int(meta['total'])
            n_paginas = int(meta['total_pages'])
        except (KeyError, ValueError, TypeError) as exc:
            raise ValueError('Metadatos de página, límite o total incorrectos') from exc
        if n_pagina != pagina or limite != REGISTROS_POR_PAGINA_CHILE:
            raise ValueError('Página o tamaño de página distintos de la solicitud')
        if n_paginas < 1 or n_total < 1:
            raise ValueError('Cantidad de páginas o registros no válida')
        if esperado is not None and n_paginas != esperado:
            raise ValueError('El total de páginas cambió durante la descarga')
        if total_esperado is not None and n_total != total_esperado:
            raise ValueError('La cantidad total de registros cambió durante la descarga')
        n_esperado = min(REGISTROS_POR_PAGINA_CHILE,
                          max(0, n_total - (pagina-1)*REGISTROS_POR_PAGINA_CHILE))
        if n_esperado <= 0 or len(registros) != n_esperado:
            raise ValueError(f'Registros insuficientes en página {pagina}: {len(registros)}/{n_esperado}')
        return {'page': n_pagina, 'limit': limite, 'total': n_total, 'total_pages': n_paginas}

    if destino.exists() and not REFRESCAR_DOCUMENTOS:
        try:
            guardado = json.loads(destino.read_text(encoding='utf-8-sig'))
            meta = validar_json(guardado)
            return ({'codigo': codigo, 'pagina': pagina, 'estado': 'EXISTENTE_VALIDADO',
                     'n': len(guardado['data']), 'total': meta['total'],
                     'paginas_totales': meta['total_pages'], 'archivo': str(destino),
                     'url': url, 'sha256': sha256_archivo(destino), 'http': 200,
                     'intentos': 0, 'error': ''}, guardado)
        except (ValueError, KeyError, TypeError, UnicodeError, OSError):
            # Volvemos a consultar las páginas que faltan o no pasan la validación.
            pass

    error_final = ''
    http_final = None
    for intento in range(1, REINTENTOS_CHILE+1):
        if time.monotonic() - _INICIO_DESCARGA_CHILE >= PRESUPUESTO_CHILE_SEG:
            error_final = 'Presupuesto de reintentos agotado; se conserva lo descargado para la siguiente sesión'
            break
        respuesta = None
        try:
            espaciar_consultas_chile()
            # Controlamos de forma explícita los reintentos por HTTP 429.
            with requests.Session() as sesion:
                sesion.headers.update({
                    'User-Agent': 'UNCP-Estudio-Conectividad-Portuaria/1.0 (investigacion)',
                    'Accept': 'application/json'})
                respuesta = sesion.get(url, params=params, timeout=TIMEOUT_HTTP)
                http_final = respuesta.status_code
                if respuesta.status_code == 429:
                    espera = pausa_entre_reintentos_chile(intento, respuesta, por_tasa=True)
                    error_final = 'HTTP 429: límite temporal de solicitudes del servidor chileno'
                    restante = PRESUPUESTO_CHILE_SEG - (time.monotonic() - _INICIO_DESCARGA_CHILE)
                    excede_presupuesto = espera > restante
                    registrar_intento_chile(codigo, pagina, intento, 429,
                                           'ESPERA_SUPERA_TIEMPO_DISPONIBLE' if excede_presupuesto else 'LIMITADO',
                                           espera, error_final)
                    print(f'  Chile {codigo}, página {pagina}: HTTP 429; pausa indicada {espera:.0f} s (intento {intento}/{REINTENTOS_CHILE}).')
                    if excede_presupuesto:
                        error_final += f'; reintentar después de {espera:.0f} s según la respuesta del servidor'
                        break
                    if intento < REINTENTOS_CHILE:
                        time.sleep(espera)
                    continue
                if respuesta.status_code in (408, 500, 502, 503, 504):
                    espera = pausa_entre_reintentos_chile(intento, respuesta)
                    error_final = f'HTTP {respuesta.status_code}, error temporal del servicio'
                    registrar_intento_chile(codigo, pagina, intento, respuesta.status_code,
                                            'TEMPORAL', espera, error_final)
                    if intento < REINTENTOS_CHILE:
                        time.sleep(espera)
                    continue
                respuesta.raise_for_status()  # Otros 4xx no se vuelven a solicitar repetidamente.
                contenido = respuesta.json()
                meta = validar_json(contenido)
            # Guardamos primero un archivo temporal para evitar páginas incompletas.
            temporal = destino.with_suffix('.json.part')
            temporal.write_text(json.dumps(contenido, ensure_ascii=False, indent=2), encoding='utf-8')
            temporal.replace(destino)
            registrar_intento_chile(codigo, pagina, intento, 200, 'VALIDADO', 0, 'Página descargada y verificada')
            return ({'codigo': codigo, 'pagina': pagina, 'estado': 'DESCARGADO_VALIDADO',
                     'n': len(contenido['data']), 'total': meta['total'],
                     'paginas_totales': meta['total_pages'], 'archivo': str(destino),
                     'url': respuesta.url, 'sha256': sha256_archivo(destino), 'http': 200,
                     'intentos': intento, 'error': ''}, contenido)
        except Exception as exc:
            error_final = f'{type(exc).__name__}: {str(exc)[:190]}'
            es_4xx_permanente = bool(respuesta is not None and
                                      400 <= respuesta.status_code < 500 and
                                      respuesta.status_code not in (408, 429))
            espera = 0 if es_4xx_permanente or intento == REINTENTOS_CHILE else pausa_entre_reintentos_chile(intento)
            registrar_intento_chile(codigo, pagina, intento, http_final,
                                    'ERROR_SIN_REINTENTO' if es_4xx_permanente else 'ERROR_TEMPORAL',
                                    espera, error_final)
            if es_4xx_permanente:
                break
            if espera > 0:
                time.sleep(espera)
    return ({'codigo': codigo, 'pagina': pagina, 'estado': 'ERROR', 'n': 0,
             'archivo': str(destino), 'url': url, 'sha256': '', 'http': http_final,
             'intentos': intento, 'error': error_final}, None)

def descargar_chile_completo(codigo):
    primero, respuesta = obtener_pagina_chile(codigo, 1)
    REGISTRO_CHILE_PAGINAS.append(primero)
    if respuesta is None:
        return {'codigo': codigo, 'estado': 'ERROR_PRIMERA_PAGINA',
                'paginas_recibidas': 0, 'paginas_totales_reportadas': None,
                'registros': 0, 'total_registros_reportados': None,
                'paginas_faltantes': '1 (no fue posible consultar los metadatos)',
                'error': primero.get('error', 'Sin acceso a la API')}

    meta = respuesta['pagination']
    total_paginas = int(meta['total_pages'])
    total_registros = int(meta['total'])
    if total_paginas < 1 or total_paginas > 50000:
        raise ValueError('El total de páginas informado por la API es inusual')

    resultados = {1: respuesta}
    pendientes = list(range(2, total_paginas + 1))
    print(f'{codigo}: {total_paginas} páginas y {total_registros:,} registros declarados por la API.')
    # Comprobamos las páginas guardadas antes de consultar la API.
    for posicion, pagina in enumerate(pendientes, start=2):
        registro, contenido = obtener_pagina_chile(codigo, pagina, total_paginas, total_registros)
        REGISTRO_CHILE_PAGINAS.append(registro)
        if contenido is not None:
            resultados[pagina] = contenido
        if posicion % 15 == 0 or posicion == total_paginas:
            print(f'  {codigo}: {len(resultados)}/{total_paginas} páginas validadas; revisadas {posicion}/{total_paginas}.')
        # Revisamos toda la carpeta aunque se detengan temporalmente las solicitudes.

    faltantes = sorted(set(range(1, total_paginas+1)) - set(resultados))
    datos = [item for pagina in sorted(resultados) for item in resultados[pagina]['data']]
    completos = not faltantes and len(datos) == total_registros
    if completos:
        df = pd.json_normalize(datos)
        df['codigo_indicador'] = codigo
        df['url_api'] = FUENTES_URL['chile_api'].rstrip('/')+f'/{codigo}/data'
        salida = SUBCARPETAS['tablas']/f'CHILE_{codigo}_2025'
        df.to_csv(salida.with_suffix('.csv'), index=False, encoding='utf-8-sig')
        df.to_excel(salida.with_suffix('.xlsx'), index=False)
        print(f'  {codigo}: registros completos, exportados a CSV y Excel ({len(df):,}).')
    errores = [x['error'] for x in REGISTRO_CHILE_PAGINAS
               if x.get('codigo') == codigo and x.get('estado') == 'ERROR']
    return {'codigo': codigo, 'estado': 'COMPLETO' if completos else 'INCOMPLETO',
            'paginas_recibidas': len(resultados), 'paginas_totales_reportadas': total_paginas,
            'registros': len(datos), 'total_registros_reportados': total_registros,
            'paginas_faltantes': ','.join(map(str, faltantes)),
            'error': '; '.join(errores)[:1000]}

resumen_chile = []
for codigo in ('P048', 'P012'):
    resultado = descargar_chile_completo(codigo)
    resumen_chile.append(resultado)
    print(f"{codigo}: {resultado['estado']} | registros {resultado['registros']}/{resultado['total_registros_reportados']} | páginas {resultado['paginas_recibidas']}/{resultado['paginas_totales_reportadas']}")

pd.DataFrame(resumen_chile).to_csv(
    SUBCARPETAS['control']/'04_DESCARGAS_CHILE.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(REGISTRO_CHILE_PAGINAS).to_csv(
    SUBCARPETAS['control']/'04A_CHILE_TRAZABILIDAD_POR_PAGINA.csv', index=False, encoding='utf-8-sig')
if any(r['estado'] != 'COMPLETO' for r in resumen_chile):
    for registro in resumen_chile:
        if registro['estado'] != 'COMPLETO':
            print(f"{registro['codigo']}: faltan las páginas {registro['paginas_faltantes']}.")
    print('Las páginas pendientes quedan registradas y pueden volver a solicitarse más adelante.')
    if EXIGIR_CHILE_COMPLETO:
        raise RuntimeError('Chile incompleto: revisar 04_DESCARGAS_CHILE.csv')


## 5. Extracción y control documental

Los documentos PDF se examinan para identificar tablas y texto estructurado. Cuando el contenido se encuentra incorporado como imagen, el reconocimiento óptico de caracteres se realiza mediante Tesseract, siempre que esté disponible. Los resultados procedentes de OCR conservan información de página, posición y confianza para su contraste con el documento original.

Los registros derivados se identifican según el procedimiento de extracción utilizado. Las cifras obtenidas mediante OCR **no se incorporan automáticamente** a la base econométrica. Si el motor de reconocimiento no está disponible, el estado documental se registra como pendiente, sin modificar las estimaciones del PLSCI.

In [ ]:
REGISTRO_EXTRACCION = []

# Buscamos Tesseract en Windows, Anaconda y las rutas del sistema.
def buscar_tesseract():
    candidatos = [os.environ.get('TESSERACT_CMD', '').strip(), shutil.which('tesseract')]
    carpetas = [Path(sys.prefix)]
    if os.environ.get('CONDA_PREFIX'):
        carpetas.append(Path(os.environ['CONDA_PREFIX']))
    for raiz in carpetas:
        candidatos.extend([raiz / 'Library' / 'bin' / 'tesseract.exe',
                           raiz / 'Scripts' / 'tesseract.exe', raiz / 'bin' / 'tesseract'])
    for entorno in ('ProgramFiles', 'ProgramFiles(x86)', 'LOCALAPPDATA'):
        if os.environ.get(entorno):
            candidatos.append(Path(os.environ[entorno]) / 'Tesseract-OCR' / 'tesseract.exe')
    candidatos.append(Path(r'C:\Program Files\Tesseract-OCR\tesseract.exe'))
    for candidato in candidatos:
        if not candidato:
            continue
        ubicacion = Path(str(candidato))
        if ubicacion.is_file():
            try:
                proceso = subprocess.run([str(ubicacion), '--version'], capture_output=True,
                                         timeout=8, check=False)
                if proceso.returncode == 0:
                    return str(ubicacion)
            except (OSError, subprocess.TimeoutExpired):
                pass
    return None

try:
    try:
        import pymupdf as fitz
    except ImportError:
        import fitz
    from PIL import Image
    import pytesseract
    from pytesseract import Output
    TESSERACT_EJECUTABLE = buscar_tesseract()
    TESSERACT_ACTIVO = bool(TESSERACT_EJECUTABLE)
    if TESSERACT_ACTIVO:
        pytesseract.pytesseract.tesseract_cmd = TESSERACT_EJECUTABLE
        IDIOMAS_OCR = pytesseract.get_languages(config='')
    else:
        IDIOMAS_OCR = []
    LENGUA_OCR = ('spa+eng' if {'spa','eng'}.issubset(IDIOMAS_OCR) else
                  'spa' if 'spa' in IDIOMAS_OCR else 'eng')
except (ImportError, OSError, RuntimeError) as exc:
    TESSERACT_ACTIVO = False
    TESSERACT_EJECUTABLE = None
    LENGUA_OCR = 'eng'
    print('No se pudo preparar el reconocimiento de imágenes:', str(exc)[:150])
print('OCR disponible:', TESSERACT_ACTIVO and USAR_OCR_PDF,
      '| ejecutable:', TESSERACT_EJECUTABLE, '| idioma:', LENGUA_OCR)
if USAR_OCR_PDF and not TESSERACT_ACTIVO:
    print('Reconocimiento OCR pendiente: no se detectó un ejecutable de Tesseract.')

def celda_pdf(x):
    if x is None: return ''
    valor=str(x).replace('\n',' ').strip()
    # Tratamos el texto extraído como contenido, no como una fórmula de Excel.
    return "'"+valor if valor.startswith(('=','+','@')) else valor

def ocr_por_pagina(pdf_fitz, numero):
    pagina = pdf_fitz[numero]
    aumento = 2.5
    pix = pagina.get_pixmap(matrix=fitz.Matrix(aumento, aumento), alpha=False)
    imagen = Image.open(io.BytesIO(pix.tobytes('png')))
    resultado = pytesseract.image_to_data(imagen, lang=LENGUA_OCR, output_type=Output.DICT,
                                         config='--psm 11')
    grupos = {}
    palabras = []
    for k, valor in enumerate(resultado['text']):
        palabra = str(valor).strip()
        if not palabra:
            continue
        try:
            confianza = float(resultado['conf'][k])
        except (TypeError, ValueError):
            confianza = -1.0
        if confianza < 15:
            continue
        clave = (int(resultado['block_num'][k]), int(resultado['par_num'][k]),
                 int(resultado['line_num'][k]))
        x0 = float(resultado['left'][k]) / aumento
        y0 = float(resultado['top'][k]) / aumento
        x1 = x0 + float(resultado['width'][k]) / aumento
        y1 = y0 + float(resultado['height'][k]) / aumento
        grupos.setdefault(clave, []).append((y0, x0, x1, y1, palabra, confianza))
        palabras.append([numero+1, palabra, round(x0, 1), round(y0, 1),
                         round(x1, 1), round(y1, 1), round(confianza, 1)])
    lineas = []
    for n, (clave, grupo) in enumerate(sorted(grupos.items(),
                                              key=lambda z: (min(w[0] for w in z[1]),
                                                             min(w[1] for w in z[1]))), 1):
        grupo.sort(key=lambda w: w[1])
        texto = ' '.join(w[4] for w in grupo)
        confianza_media = sum(w[5] for w in grupo) / len(grupo)
        lineas.append([numero+1, n, texto, round(confianza_media, 1), len(grupo),
                       round(min(w[1] for w in grupo), 1),
                       round(min(w[0] for w in grupo), 1),
                       round(max(w[2] for w in grupo), 1),
                       round(max(w[3] for w in grupo), 1)])
    return lineas, palabras

def convertir_pdf_a_excel(archivo):
    archivo=Path(archivo)
    salida=SUBCARPETAS['tablas']/(archivo.stem+'_TABLAS.xlsx')
    doc_hash=sha256_archivo(archivo)
    # Las hojas ya extraídas se conservan si corresponden al mismo PDF.
    if salida.exists() and not REFRESCAR_DOCUMENTOS:
        try:
            previo = load_workbook(salida, read_only=True, data_only=True)
            metadata = dict((str(row[0]), str(row[1])) for row in
                            previo['FUENTE'].iter_rows(min_row=1, max_col=2, values_only=True)
                            if row[0] is not None)
            otras_hojas = [x for x in previo.sheetnames if x != 'FUENTE']
            previo.close()
            if metadata.get('sha256_pdf') == doc_hash and otras_hojas:
                return {'documento': archivo.name, 'estado': 'EXCEL_VERIFICADO_REUTILIZADO',
                        'excel': str(salida), 'sha256_fuente': doc_hash,
                        'detalle': 'La hoja extraída corresponde a la misma versión del PDF'}
        except (OSError, KeyError, ValueError):
            pass
    info=(next((r for r in REGISTRO_DESCARGAS + DESCARGAS_PREVIAS
            if str(r.get('archivo','')).replace('\\','/').split('/')[-1]==archivo.name),{}))
    libro=Workbook()
    fuente=libro.active;fuente.title='FUENTE'
    for k,v in [ ('documento_original',archivo.name),('url_origen',info.get('url_final',info.get('url_solicitada',''))),
                 ('sha256_pdf',doc_hash),('metodos','tablas PDF / texto PDF / OCR'),
                 ('observacion','Cifras detectadas por OCR requieren revisión con la página original')]:
        fuente.append([k,str(v)])
    total_paginas=0;paginas_procesadas=0;tablas=0;lineas_texto=0;lineas_ocr=0
    sin_ocr=0
    hoja_ocr=libro.create_sheet('OCR_LINEAS')
    hoja_ocr.append(['pagina','linea','texto_transcrito','confianza_media_pct','palabras',
                     'x0_pt','y0_pt','x1_pt','y1_pt'])
    hoja_palabras=libro.create_sheet('OCR_PALABRAS')
    hoja_palabras.append(['pagina','palabra','x0_pt','y0_pt','x1_pt','y1_pt','confianza_pct'])
    hoja_texto=libro.create_sheet('TEXTO_PDF')
    hoja_texto.append(['pagina','linea','texto_de_la_pagina'])
    try:
        with pdfplumber.open(archivo) as pdf:
            total_paginas=len(pdf.pages)
            # Para Ecuador extraemos la página que contiene la tabla 5.
            alcance_dirigido=(archivo.name=='BOLETIN_PORTUARIO_2025.pdf' and total_paginas>=16)
            indices=([15] if alcance_dirigido else
                     range(total_paginas if PAGINAS_PDF_A_EXAMINAR<=0 else
                           min(total_paginas,PAGINAS_PDF_A_EXAMINAR)))
            imagenes_pdf=None
            for idx in indices:
                page=pdf.pages[idx]
                paginas_procesadas+=1
                tablas_pagina=page.extract_tables() or []
                for t,tabla in enumerate(tablas_pagina,start=1):
                    if not tabla or len(tabla)<2:continue
                    tablas+=1
                    hoja=libro.create_sheet(f'P{idx+1:03d}_T{t:02d}'[:31])
                    hoja.append(['pagina_pdf',idx+1,'tabla_numero',t])
                    for row in tabla:
                        hoja.append([celda_pdf(c) for c in row])
                texto=page.extract_text() or ''
                if texto.strip():
                    for n,linea in enumerate(texto.splitlines(),start=1):
                        if linea.strip():
                            hoja_texto.append([idx+1,n,celda_pdf(linea)])
                            lineas_texto+=1
                # Aplicamos OCR cuando la página no contiene tablas de texto.
                if not tablas_pagina and USAR_OCR_PDF and (total_paginas <= 3 or len(texto.strip()) < 25):
                    if TESSERACT_ACTIVO:
                        if imagenes_pdf is None: imagenes_pdf=fitz.open(archivo)
                        lineas, palabras = ocr_por_pagina(imagenes_pdf, idx)
                        for row in lineas:
                            hoja_ocr.append(row)
                            lineas_ocr += 1
                        for row in palabras:
                            hoja_palabras.append(row)
                    elif not texto.strip():
                        sin_ocr+=1
            if imagenes_pdf is not None: imagenes_pdf.close()
        # Retiramos las hojas sin información.
        if lineas_ocr==0:
            libro.remove(hoja_ocr)
            libro.remove(hoja_palabras)
        if lineas_texto==0:libro.remove(hoja_texto)
        if tablas or lineas_texto or lineas_ocr:
            libro.save(salida)
            estado=('TABLAS_ESTRUCTURADAS' if tablas else
                    'TEXTO_Y_OCR' if lineas_ocr and lineas_texto else
                    'OCR_TRANSCRIPCION_REVISAR' if lineas_ocr else 'SOLO_TEXTO')
            if paginas_procesadas<total_paginas:
                estado+=('_EXTRACCION_DIRIGIDA_TABLA5' if alcance_dirigido else '_COBERTURA_PARCIAL')
            if sin_ocr:estado+='_OCR_PENDIENTE'
            ruta_excel=str(salida)
        else:
            estado=('OCR_SIN_EJECUTABLE' if USAR_OCR_PDF and not TESSERACT_ACTIVO else
                    'REQUIERE_OCR_O_DOCUMENTO_ILEGIBLE')
            ruta_excel=''
        return {'documento':archivo.name,'paginas':paginas_procesadas,
                'paginas_totales':total_paginas,'tablas':tablas,'lineas_texto':lineas_texto,
                'lineas_ocr':lineas_ocr,'paginas_pendientes_ocr':sin_ocr,
                'estado':estado,'excel':ruta_excel,'sha256_fuente':doc_hash,
                'url_fuente':info.get('url_final',info.get('url_solicitada',''))}
    except Exception as exc:
        return {'documento':archivo.name,'estado':'ERROR_EXTRACCION',
                'detalle':str(exc)[:260], 'sha256_fuente':doc_hash}

# Revisamos también los PDF que permanecen en la carpeta de fuentes.
pdf_candidatos=[Path(x['archivo']) for x in REGISTRO_DESCARGAS
                if x['estado']!='ERROR' and x['tipo_esperado']=='pdf']
pdf_candidatos+=list(SUBCARPETAS['fuentes'].rglob('*.pdf'))
pdf_descargados=sorted({str(p):p for p in pdf_candidatos
                        if p.is_file() and validar_tipo(p,'pdf')}.values(),key=str)
print('Convirtiendo',len(pdf_descargados),'PDF oficiales, con',WORKERS_PDF,'trabajadores...')
with ThreadPoolExecutor(max_workers=WORKERS_PDF) as pool:
    trabajos={pool.submit(convertir_pdf_a_excel,p):p for p in pdf_descargados}
    for futuro in as_completed(trabajos):
        result=futuro.result()
        REGISTRO_EXTRACCION.append(result)
        print(result.get('documento','')[:52],result['estado'],
              'tablas:',result.get('tablas',0),'OCR:',result.get('lineas_ocr',0))

excel_candidatos=[Path(x['archivo']) for x in REGISTRO_DESCARGAS
                    if x['estado']!='ERROR' and x['tipo_esperado']=='xlsx']
excel_candidatos+=list(SUBCARPETAS['fuentes'].rglob('*.xlsx'))
exceles_descargados=sorted({str(p):p for p in excel_candidatos
                           if p.is_file() and validar_tipo(p,'xlsx')}.values(),key=str)
for archivo in exceles_descargados:
    try:
        origen=load_workbook(archivo,read_only=True,data_only=True)
        destino=SUBCARPETAS['tablas']/(archivo.stem+'_HOJA_ORIGINAL.xlsx')
        salida=Workbook();salida.remove(salida.active)
        for hojas in origen.worksheets:
            nueva=salida.create_sheet(hojas.title[:31])
            for fila in hojas.iter_rows(values_only=True):
                nueva.append([celda_pdf(v) if isinstance(v,str) else v for v in fila])
        origen.close();salida.save(destino)
        REGISTRO_EXTRACCION.append({'documento':archivo.name,'estado':'XLSX_OFICIAL_IMPORTADO',
                                   'hojas':len(salida.worksheets),'excel':str(destino),
                                   'sha256_fuente':sha256_archivo(archivo)})
    except Exception as exc:
        REGISTRO_EXTRACCION.append({'documento':archivo.name,'estado':'ERROR_EXCEL',
                                   'detalle':str(exc)[:200]})

colombia_path=SUBCARPETAS['fuentes']/'COLOMBIA'/'TRAFICO_PORTUARIO_SOCRATA.csv'
if colombia_path.exists():
    colombia=pd.read_csv(colombia_path,low_memory=False)
    colombia.to_excel(SUBCARPETAS['tablas']/'COLOMBIA_SOCRATA.xlsx',index=False)
    print('Colombia:',len(colombia),'registros CSV convertidos a Excel')
pd.DataFrame(REGISTRO_EXTRACCION).to_csv(
    SUBCARPETAS['control']/'05_CONVERSION_DOCUMENTOS.csv',index=False,encoding='utf-8-sig')
print('Documentos originales examinados:',len(REGISTRO_EXTRACCION))

### 5.1. Tabla de referencia del boletín ecuatoriano

Del boletín estadístico portuario de Ecuador correspondiente a 2025 se extrae la tabla 5, situada en la página impresa 16. La extracción conserva los seis registros publicados, incluido el total, y documenta la procedencia del archivo y de la página. Los valores se mantienen en sus unidades originales y se archivan únicamente como referencia estadística.

In [ ]:
ecuador=pd.DataFrame()
PDF_ECUADOR=SUBCARPETAS['fuentes']/'ECUADOR'/'BOLETIN_PORTUARIO_2025.pdf'
if PDF_ECUADOR.exists():
    registros=[]
    try:
        with pdfplumber.open(PDF_ECUADOR) as doc:
            if len(doc.pages)<16: raise ValueError('El boletín tiene menos de 16 páginas')
            tablas=doc.pages[15].extract_tables() or []
        for tabla in tablas:
            encontrados=[]
            for fila in tabla:
                celdas=[str(v).replace('\n',' ').strip() for v in fila if v and str(v).strip()]
                puerto=next((v for v in celdas if v in {'APE','APM','APG','APPB','TPH','TOTAL'}),None)
                if puerto is None:continue
                nums=[v for v in celdas if re.fullmatch(r'\d[\d.]*%?',v)]
                if len(nums)==5:
                    encontrados.append({'anio':2025,'codigo_grupo_portuario':puerto,
                        'importacion_teu':int(nums[0].replace('.','')),
                        'exportacion_teu':int(nums[1].replace('.','')),
                        'total_teu_publicado':int(nums[2].replace('.','')),
                        'participacion_importacion_pct':int(nums[3].replace('%','')),
                        'participacion_exportacion_pct':int(nums[4].replace('%','')),
                        'pagina_pdf':16,'tabla':5,'sha256_fuente':sha256_archivo(PDF_ECUADOR)})
            if len(encontrados)==6:
                registros=encontrados;break
        if not registros: raise ValueError('No se encontraron las seis filas de la tabla 5')
        ecuador=pd.DataFrame(registros)
        ecuador.to_excel(SUBCARPETAS['tablas']/'ECUADOR_TABLA5_2025.xlsx',index=False)
        print('Ecuador: seis filas extraídas directamente del PDF y exportadas a Excel')
        display(ecuador[['codigo_grupo_portuario', 'total_teu_publicado']])
    except Exception as exc:
        print('La tabla ecuatoriana requiere revisar el documento:',str(exc)[:240])
else:
    print('El boletín de Ecuador no se descargó; se omite su conversión específica.')

## 6. Construcción de la base econométrica

Se seleccionan los catorce códigos portuarios y los siete trimestres definidos para el estudio. La depuración comprueba la unicidad de la llave puerto–trimestre, la cobertura completa de las 98 combinaciones y la positividad del PLSCI. A continuación, se calculan el logaritmo natural del índice, los indicadores portuarios y temporales, y la tendencia centrada en 2025T4.

La base se exporta en CSV y Excel a `BASE_FINAL`, junto con su diccionario de variables y los metadatos de procedencia. **Todas las estimaciones posteriores se realizan a partir del CSV final reimportado**, cuyo contenido se contrasta con los registros seleccionados de UNCTAD.

In [ ]:
serie_original=pd.read_csv(PLSCI_ORIGEN,low_memory=False,
    dtype={'Quarter':'string','Port':'string'})
col_valor='Index (average Q1 2023 = 100)'
requeridas=['Quarter','Port','Port Label',col_valor]
if not set(requeridas).issubset(serie_original.columns):
    raise ValueError('Las columnas oficiales de UNCTAD han cambiado: '+str(serie_original.columns.tolist()))

mapa_periodos={p.replace('T','Q0'):p for p in PERIODOS}
panel=(serie_original.loc[
    serie_original['Quarter'].isin(mapa_periodos)&serie_original['Port'].isin(CODIGOS),
    requeridas].copy())
panel.columns=['trimestre_unctad','codigo','nombre_unctad','plsci']
panel['periodo']=panel['trimestre_unctad'].map(mapa_periodos)
panel['puerto']=panel['codigo'].map(NOMBRE_PUERTO)
panel['pais']=panel['codigo'].str[:2].map({'PE':'Perú','CL':'Chile','CO':'Colombia','EC':'Ecuador'})
panel['tau']=panel['periodo'].map({p:i-3 for i,p in enumerate(PERIODOS)})
panel['plsci']=pd.to_numeric(panel['plsci'],errors='raise')
panel=panel.sort_values(['codigo','periodo']).reset_index(drop=True)
if (len(panel)!=98 or panel.codigo.nunique()!=14 or panel.periodo.nunique()!=7
    or panel.duplicated(['codigo','periodo']).any()
    or panel.plsci.isna().any() or not (panel.plsci>0).all()):
    raise AssertionError('La base no cumple las condiciones del panel balanceado 14 x 7')
panel['ln_plsci']=np.log(panel.plsci)
panel['chancay_post']=((panel.codigo=='PECHY')&
                      (panel.periodo.isin(PERIODOS[2:]))).astype(int)
panel['url_fuente']=FUENTES_URL['unctad_plsci']
# Guardamos el panel depurado antes de cualquier estimación.
BASE_FINAL_CSV=SUBCARPETAS['base_final']/'BASE_FINAL_PLSCI_14_PUERTOS_2025T1_2026T3.csv'
BASE_FINAL_XLSX=SUBCARPETAS['base_final']/'BASE_FINAL_PLSCI_14_PUERTOS_2025T1_2026T3.xlsx'
columnas_base=['codigo','puerto','pais','trimestre_unctad','periodo',
               'tau','plsci','ln_plsci','chancay_post','url_fuente']
base_final_original=panel[columnas_base].copy()
base_final_original.to_csv(BASE_FINAL_CSV,index=False,encoding='utf-8-sig',float_format='%.15g')

diccionario=pd.DataFrame([
    ('codigo','Texto','Código internacional del puerto registrado en UNCTAD.'),
    ('puerto','Texto','Nombre identificativo del puerto.'),
    ('pais','Texto','País donde se encuentra el puerto.'),
    ('trimestre_unctad','Texto','Trimestre según el formato original de UNCTAD.'),
    ('periodo','Texto','Trimestre de la muestra, desde 2025T1 hasta 2026T3.'),
    ('tau','Entero','Tendencia temporal centrada en 2025T4; toma valores de -3 a 3.'),
    ('plsci','Índice','Port Liner Shipping Connectivity Index, puntos positivos de UNCTAD.'),
    ('ln_plsci','Logaritmo','Logaritmo natural del PLSCI; variable dependiente.'),
    ('chancay_post','Indicador 0/1','Igual a 1 para Chancay en 2025T3-2026T3.'),
    ('url_fuente','URL','Servicio oficial del que se descargó el PLSCI mundial.')
],columns=['variable','tipo','definicion'])
trazabilidad=pd.DataFrame([
    ('Institución','UNCTAD'),
    ('Conjunto de datos','US.PLSCI'),
    ('Página informativa','https://unctadstat.unctad.org/datacentre/reportInfo/US.PLSCI'),
    ('Endpoint usado',FUENTES_URL['unctad_plsci']),
    ('Archivo fuente',PLSCI_ORIGEN.name),
    ('SHA-256 del CSV oficial',sha256_archivo(PLSCI_ORIGEN)),
    ('Observaciones',len(base_final_original)),
    ('Puertos',base_final_original.codigo.nunique()),
    ('Trimestres',base_final_original.periodo.nunique()),
],columns=['atributo','valor'])
with pd.ExcelWriter(BASE_FINAL_XLSX,engine='openpyxl') as escritor:
    base_final_original.to_excel(escritor,sheet_name='BASE_FINAL',index=False)
    diccionario.to_excel(escritor,sheet_name='DICCIONARIO',index=False)
    trazabilidad.to_excel(escritor,sheet_name='FUENTE_Y_PROCEDENCIA',index=False)
libro=load_workbook(BASE_FINAL_XLSX)
for hoja in libro.worksheets:
    hoja.freeze_panes='A2'
    hoja.sheet_view.showGridLines=False
    hoja.auto_filter.ref=hoja.dimensions
    for c in hoja[1]:
        c.fill=PatternFill(fill_type='solid',fgColor='183A54')
        c.font=Font(name='Calibri',bold=True,color='FFFFFF')
        c.alignment=Alignment(wrap_text=True)
    for col in hoja.columns:
        ancho=min(54,max(13,max(len(str(c.value or '')) for c in list(col)[:250])+2))
        hoja.column_dimensions[col[0].column_letter].width=ancho
# Añadimos los enlaces de las fuentes al libro de Excel.
hoja_datos=libro['BASE_FINAL']
for numero_fila in range(2,hoja_datos.max_row+1):
    celda=hoja_datos.cell(row=numero_fila,column=columnas_base.index('url_fuente')+1)
    celda.hyperlink=str(celda.value)
    celda.font=Font(name='Calibri',color='0563C1',underline='single')
libro.save(BASE_FINAL_XLSX)

# Volvemos a leer el CSV final para construir las matrices del modelo.
del panel
panel=pd.read_csv(BASE_FINAL_CSV,dtype={'codigo':'string','periodo':'string',
                                        'trimestre_unctad':'string'},encoding='utf-8-sig')
panel['tau']=panel['tau'].astype(int)
panel['chancay_post']=panel['chancay_post'].astype(int)
assert len(panel)==98 and panel.codigo.nunique()==14 and panel.periodo.nunique()==7
assert not panel.duplicated(['codigo','periodo']).any()
assert panel[['plsci','ln_plsci','tau']].notna().all().all()
assert np.allclose(panel['ln_plsci'].to_numpy(float),np.log(panel['plsci'].to_numpy(float)),atol=1e-12)
assert list(panel['codigo'].astype(str))==list(base_final_original['codigo'].astype(str))
assert np.allclose(panel['plsci'],base_final_original['plsci'],atol=1e-10)

(BASE_FINAL_CSV.parent/'METADATOS_BASE_FINAL.json').write_text(
    json.dumps({'fuente':FUENTES_URL['unctad_plsci'],
                'sha256_csv_original':sha256_archivo(PLSCI_ORIGEN),
                'sha256_base_final_csv':sha256_archivo(BASE_FINAL_CSV),
                'observaciones':len(panel),'puertos':int(panel.codigo.nunique()),
                'trimestres':int(panel.periodo.nunique()),
                'archivo_csv':BASE_FINAL_CSV.name,'archivo_excel':BASE_FINAL_XLSX.name},
               ensure_ascii=False,indent=2),encoding='utf8')
print('Base final exportada y reimportada:',BASE_FINAL_CSV)
print('Excel analítico:',BASE_FINAL_XLSX)
print('Integridad de los 98 registros verificada.')

resumen_puertos=(panel.groupby(['codigo','puerto','pais'],as_index=False)
    .agg(observaciones=('plsci','size'),pl_sci_2025t1=('plsci','first'),
         pl_sci_2026t3=('plsci','last')))
resumen_puertos['variacion_observada_pct']=100*(resumen_puertos.pl_sci_2026t3 /
                                                resumen_puertos.pl_sci_2025t1-1)
resumen_puertos.to_csv(SUBCARPETAS['datos']/'RESUMEN_PUERTOS.csv',index=False,encoding='utf-8-sig')
print('Observaciones:',len(panel),'| Puertos:',panel.codigo.nunique(),
      '| Trimestres:',panel.periodo.nunique())
print('Rango de valores PLSCI:',round(panel.plsci.min(),4),round(panel.plsci.max(),4))
display(resumen_puertos[['puerto', 'pais', 'observaciones', 'variacion_observada_pct']].round(2))

## 7. Especificación econométrica

Se denota por $i=1,\ldots,N$ el puerto y por $t=1,\ldots,T$ el trimestre, con $N=14$ y $T=7$. La variable dependiente es el logaritmo natural del índice de conectividad, mientras que la tendencia centrada toma valores enteros de $-3$ a $3$:

$$
\begin{aligned}
y_{it}&=\ln\!\left(\mathrm{PLSCI}_{it}\right),\\
\tau_t&\in\{-3,-2,-1,0,1,2,3\},\qquad \tau_{2025\mathrm{T}4}=0.
\end{aligned}
$$

**Modelo principal.** Se incorporan efectos fijos de puerto y de trimestre, así como tendencias específicas por puerto, tomando Callao como referencia:

$$
y_{it}=\alpha_i+\lambda_t+
\sum_{j\neq\mathrm{Callao}}\gamma_j\,D_{ij}\tau_t+\varepsilon_{it}
\tag{1}
$$

En esta expresión, $D_{ij}$ indica si la observación corresponde al puerto $j$; $\alpha_i$ recoge diferencias permanentes entre puertos, $\lambda_t$ representa los cambios trimestrales comunes y $\gamma_j$ cuantifica la diferencia de tendencia respecto de Callao. La omisión de la interacción correspondiente a Callao fija la referencia de las pendientes: **no supone que su conectividad sea constante**.

Para Chancay y Callao, la diferencia logarítmica del mismo trimestre es

$$
\ln\!\left(\frac{\mathrm{PLSCI}_{\mathrm{Chancay},t}}
{\mathrm{PLSCI}_{\mathrm{Callao},t}}\right)
=\bigl(\alpha_{\mathrm{Chancay}}-\alpha_{\mathrm{Callao}}\bigr)
+\gamma_{\mathrm{Chancay}}\tau_t
+\bigl(\varepsilon_{\mathrm{Chancay},t}-\varepsilon_{\mathrm{Callao},t}\bigr).
\tag{2}
$$

La variación porcentual trimestral estimada de la **razón entre índices**, y no de los niveles absolutos de los puertos, se obtiene como

$$
g_{\mathrm{Chancay/Callao}}=100\left[\exp\!\left(\gamma_{\mathrm{Chancay}}\right)-1\right].
\tag{3}
$$

**Especificación ampliada.** Se considera una interacción de Chancay con un indicador de periodo posterior, igual a uno desde 2025T3:

$$
y_{it}=\alpha_i+\lambda_t+
\sum_{j\neq\mathrm{Callao}}\gamma_jD_{ij}\tau_t
+\delta\,D_{i,\mathrm{Chancay}}\,\mathrm{Post}_t+\varepsilon_{it}
\tag{4}
$$

El coeficiente $\delta$ representa un cambio adicional de nivel respecto de la tendencia específica. Se estima, además, una especificación base que omite las tendencias individuales y un modelo complementario de variaciones logarítmicas:

$$
\begin{aligned}
\text{Modelo base:}\quad &y_{it}=\alpha_i+\lambda_t+\varepsilon_{it},\\[3pt]
\text{Crecimiento trimestral:}\quad
&\Delta\ln\!\left(\mathrm{PLSCI}_{it}\right)=\mu_i+\xi_t+u_{it},
\qquad t=2,\ldots,T.
\end{aligned}
\tag{5}
$$

El modelo en variaciones explica un resultado distinto del de las especificaciones en niveles y se interpreta por separado.

In [ ]:
# Las estimaciones utilizan el CSV guardado en BASE_FINAL.
assert BASE_FINAL_CSV.exists(), 'Primero debe exportarse BASE_FINAL'
assert sha256_archivo(BASE_FINAL_CSV)==json.loads((SUBCARPETAS['base_final']/'METADATOS_BASE_FINAL.json').read_text(encoding='utf8'))['sha256_base_final_csv']

def estimar_panel(datos, tendencias=True, incluir_dummy=False, dependiente="ln_plsci"):

    tabla = datos.reset_index(drop=True).copy()
    efectos_puerto = pd.get_dummies(tabla.codigo, dtype=float).drop(columns=["PECLL"])
    efectos_trimestre = pd.get_dummies(tabla.periodo, dtype=float).iloc[:, 1:]
    X = pd.concat([
        pd.Series(1.0, index=tabla.index, name="constante"),
        efectos_puerto.add_prefix("puerto_"),
        efectos_trimestre.add_prefix("trimestre_")
    ], axis=1)
    if tendencias:
        interacciones = efectos_puerto.mul(tabla.tau, axis=0).add_prefix("tendencia_")
        X = pd.concat([X, interacciones], axis=1)
    if incluir_dummy:
        X["chancay_post"] = tabla.chancay_post.astype(float)
    if np.linalg.matrix_rank(X.to_numpy()) != X.shape[1]:
        raise ValueError("La matriz de regresores presenta dependencia lineal")
    modelo = ModeloMCO(tabla[dependiente].to_numpy(), X).fit()
    return modelo

modelo_base = estimar_panel(panel, tendencias=False)
modelo_principal = estimar_panel(panel, tendencias=True)
modelo_dummy = estimar_panel(panel, tendencias=True, incluir_dummy=True)
panel["variacion_log_trimestral"] = panel.groupby("codigo").ln_plsci.diff()
modelo_crecimiento = estimar_panel(panel.dropna(subset=["variacion_log_trimestral"]),
                                  tendencias=False, dependiente="variacion_log_trimestral")
MODELOS = {
    "EF puerto y trimestre": modelo_base,
    "Modelo 1 principal (EF + tendencias)": modelo_principal,
    "Extensión con dummy Chancay": modelo_dummy,
    "Crecimiento log trimestral": modelo_crecimiento,
}

gamma_chancay = float(modelo_principal.params["tendencia_PECHY"])
assert abs(gamma_chancay - 0.0714952613307549) < 1e-9, 'La tendencia no coincide con el resultado publicado; revisar la versión PLSCI descargada.'
assert abs(float(modelo_dummy.params['chancay_post']) - 0.0054228503) < 1e-8, 'La estimación de la dummy difiere de la del artículo; revisar los datos.'
assert abs(modelo_principal.rsquared_adj - 0.990618757) < 1e-7, 'El R2 ajustado difiere del artículo; revisar el panel.'

print("Coeficiente tendencia Chancay respecto de Callao:", round(gamma_chancay, 9))
print("Variación equivalente (% por trimestre):", round(100 * np.expm1(gamma_chancay), 4))
print("R² ajustado del Modelo 1:", round(modelo_principal.rsquared_adj, 6))
print("Cambio de nivel adicional con dummy:", round(float(modelo_dummy.params["chancay_post"]), 6))

### 7.1. Estimación e inferencia

Los parámetros se estiman por mínimos cuadrados ordinarios a partir de las matrices de diseño explícitas. La implementación verifica rango completo y grados de libertad positivos. Se calculan errores estándar convencionales y errores HC3, además de $R^2$ ajustado, raíz del error cuadrático medio residual, AIC y BIC.

La inferencia de los contrastes de tendencia se obtiene de las regresiones auxiliares con siete observaciones trimestrales y cinco grados de libertad. Sus pruebas $t$ e intervalos al 95 % suponen perturbaciones homocedásticas y temporalmente independientes; HC3 se informa como comprobación frente a heterocedasticidad y no elimina por sí mismo posibles dependencias temporales.

In [ ]:
metricas = []
coeficientes = []
for etiqueta, regresion in MODELOS.items():
    metricas.append({"modelo": etiqueta, "n": int(regresion.nobs),
                     "parametros": len(regresion.params), "gl_residuales": int(regresion.df_resid),
                     "r2_ajustado": float(regresion.rsquared_adj),
                     "rmse_residual": float(np.sqrt(regresion.mse_resid)),
                     "aic": float(regresion.aic), "bic": float(regresion.bic)})
    hc3 = regresion.get_robustcov_results(cov_type="HC3", use_t=True)
    for i, nombre in enumerate(regresion.params.index):
        coeficientes.append({"modelo": etiqueta, "parametro": nombre,
                             "coeficiente": float(regresion.params.iloc[i]),
                             "error_ols": float(regresion.bse.iloc[i]),
                             "p_ols": float(regresion.pvalues.iloc[i]),
                             "error_hc3": float(hc3.bse[i]),
                             "p_hc3": float(hc3.pvalues[i])})
metricas = pd.DataFrame(metricas)
coeficientes = pd.DataFrame(coeficientes)
metricas.to_csv(SUBCARPETAS["resultados"] / "01_METRICAS_MODELOS.csv", index=False)
coeficientes.to_csv(SUBCARPETAS["resultados"] / "02_COEFICIENTES_COMPLETOS.csv", index=False)

panel["ajustado_ln_plsci"] = modelo_principal.fittedvalues
panel["residuo_modelo_1"] = modelo_principal.resid
panel.to_csv(SUBCARPETAS["resultados"] / "PANEL_PLSCI_CON_AJUSTES_Y_RESIDUOS.csv", index=False, encoding="utf-8-sig")
display(metricas.round(5))

## 8. Contrastes relativos y análisis de sensibilidad

La comparación principal utiliza el logaritmo de la razón de conectividad Chancay/Callao. Se incorpora una segunda referencia, constituida por la media geométrica de San Antonio, Valparaíso, Buenaventura y Guayaquil, con ponderaciones uniformes. Para este conjunto de cuatro puertos, la brecha y su tendencia se expresan como

$$
\begin{aligned}
r_t&=\ln\!\left(\mathrm{PLSCI}_{\mathrm{Chancay},t}\right)
-\frac{1}{4}\sum_{k\in\mathcal R}\ln\!\left(\mathrm{PLSCI}_{kt}\right),\\
r_t&=a+b\tau_t+v_t,\qquad
\mathcal R=\{\text{San Antonio, Valparaíso, Buenaventura, Guayaquil}\}.
\end{aligned}
\tag{6}
$$

La transformación $100[\exp(b)-1]$ expresa la variación trimestral estimada de la razón entre el índice de Chancay y la media geométrica del referente. Como comprobaciones de estabilidad se reestima el panel al retirar cada trimestre por separado y al restringir la muestra a 2025T3–2026T3. También se examina la sensibilidad de la interacción comercial al desplazar el corte temporal a 2025T4.

In [ ]:
matriz_niveles = panel.pivot(index="periodo", columns="codigo", values="plsci").reindex(PERIODOS)
matriz_logs = np.log(matriz_niveles)
centro_tiempo = np.arange(len(PERIODOS)) - 3
series_comparativas = {
    "Chancay / Callao": matriz_logs.PECHY - matriz_logs.PECLL,
    "Chancay / cuatro referentes": matriz_logs.PECHY - matriz_logs[CUATRO_REFERENTES].mean(axis=1)
}
contrastes = []
for nombre, brecha in series_comparativas.items():
    estimacion = ModeloMCO(brecha.to_numpy(), agregar_constante(centro_tiempo)).fit()
    for indice, termino in ((0, "nivel_centrado"), (1, "tendencia_trimestral")):
        limites = estimacion.conf_int()[indice]
        contrastes.append({"comparacion": nombre, "parametro": termino,
                           "coeficiente_log": float(estimacion.params[indice]),
                           "p_ols": float(estimacion.pvalues[indice]),
                           "ic95_inf": float(limites[0]), "ic95_sup": float(limites[1]),
                           "equivalente_pct": float(100 * np.expm1(estimacion.params[indice]))})
contrastes = pd.DataFrame(contrastes)
assert abs(float(contrastes.query('comparacion == "Chancay / Callao" and parametro == "tendencia_trimestral"').coeficiente_log.iloc[0]) - gamma_chancay) < 1e-9

sensibilidad = []
for trimestre in PERIODOS:
    ajuste = estimar_panel(panel[panel.periodo != trimestre])
    gamma = float(ajuste.params["tendencia_PECHY"])
    sensibilidad.append({"ejercicio": "Excluir " + trimestre, "n": int(ajuste.nobs),
                         "gamma_log": gamma, "gamma_pct": float(100 * np.expm1(gamma))})
postsolamente = estimar_panel(panel[panel.periodo >= "2025T3"])
sensibilidad.append({"ejercicio": "Solo 2025T3-2026T3", "n": int(postsolamente.nobs),
                    "gamma_log": float(postsolamente.params["tendencia_PECHY"]),
                    "gamma_pct": float(100*np.expm1(postsolamente.params["tendencia_PECHY"]))})
sensibilidad = pd.DataFrame(sensibilidad)

variacion_dummy = panel.copy()
variacion_dummy["chancay_post"] = ((variacion_dummy.codigo == "PECHY") &
                                  (variacion_dummy.periodo >= "2025T4")).astype(int)
modelo_dummy_alternativo = estimar_panel(variacion_dummy, incluir_dummy=True)

contrastes.to_csv(SUBCARPETAS["resultados"] / "03_CONTRASTES_LOGARITMICOS.csv", index=False)
sensibilidad.to_csv(SUBCARPETAS["resultados"] / "04_SENSIBILIDAD_TRIMESTRES.csv", index=False)
display(contrastes.round(5))
print("\nCambio de corte dummy a 2025T4 (%):",
      round(float(100 * np.expm1(modelo_dummy_alternativo.params["chancay_post"])), 4))

## 9. Diagnóstico espacial de los residuos

Se estudia el agrupamiento espacial residual sin modificar la especificación principal. Se construyen matrices $W=(w_{ik})$ de tres vecinos más próximos, cuatro vecinos y distancia geográfica inversa, calculada mediante Haversine. Se añaden dos matrices exploratorias derivadas del índice bilateral de UNCTAD para 2025T1 y 2026T1; estas últimas representan relaciones **entre países**, no servicios observados entre pares de puertos.

En todas las matrices la diagonal es nula y los pesos se normalizan por fila. Si $\hat\varepsilon_{it}$ es el residuo del modelo principal y $\widetilde\varepsilon_{it}$ es ese residuo centrado por trimestre, el estadístico global para el panel es

$$
I=\frac{N}{S_0}
\frac{\displaystyle\sum_{t=1}^{T}\sum_{i=1}^{N}\sum_{k=1}^{N}
 w_{ik}\widetilde\varepsilon_{it}\widetilde\varepsilon_{kt}}
{\displaystyle\sum_{t=1}^{T}\sum_{i=1}^{N}\widetilde\varepsilon_{it}^{\,2}},
\qquad S_0=\sum_{i=1}^{N}\sum_{k=1}^{N}w_{ik}.
\tag{7}
$$

La prueba global conserva las siete observaciones de cada puerto como una unidad durante las permutaciones. El resultado permite identificar posibles patrones de asociación espacial de los residuos bajo las matrices evaluadas, sin atribuirles una interpretación causal.

In [ ]:
# Coordenadas de los puertos utilizadas para calcular distancias geográficas.
GEOREFERENCIACION = [
    ("PECHY", -11.56667, -77.26667, "https://unlocode.info/PECHY"),
    ("PECLL", -12.05, -77.13333, "https://unlocode.info/PECLL"),
    ("PEPAI", -5.07666667, -81.105, "https://www.puertopaita.com/infraestructura/"),
    ("CLANF", -23.65, -70.40694444, "https://www.wikidata.org/wiki/Q111386978"),
    ("CLARI", -18.48333, -70.31667, "https://unlocode.info/CLARI"),
    ("CLCNL", -37.01667, -73.15, "https://unlocode.info/CLCNL"),
    ("CLIQQ", -20.21667, -70.13333, "https://unlocode.info/CLIQQ"),
    ("CLPAG", -23.08668, -70.41206, "https://www.geonames.org/11550061/puerto-angamos.html"),
    ("CLSAI", -33.6, -71.6, "https://unlocode.info/CLSAI"),
    ("CLSVE", -36.7, -73.13333, "https://unlocode.info/CLSVE"),
    ("CLVAP", -33.03333, -71.63333, "https://unlocode.info/CLVAP"),
    ("COBUN", 3.88333, -77.06667, "https://incodocs.com/ports/co/sea-port/cobun"),
    ("ECGYE", -2.16667, -79.9, "https://unlocode.info/ECGYE"),
    ("ECPBO", -3.26527778, -80.00027778, "https://www.puertobolivar.gob.ec/la-institucion/ubicacion/"),
]
if LSBCI_ORIGEN is None:
    raise RuntimeError('Para reproducir las matrices LSBCI es necesario descargar esta serie desde UNCTAD')
RAW_LSBCI = LSBCI_ORIGEN
ls_original = pd.read_csv(
    RAW_LSBCI, usecols=["Quarter", "Economy", "Partner", "Index"],
    dtype={"Quarter": "string", "Economy": "string", "Partner": "string"}, low_memory=False
)
CODIGOS_PAISES_UNCTAD = {"152": "CHILE", "170": "COLOMBIA", "218": "ECUADOR", "604": "PERU"}
sel = ls_original.loc[
    ls_original.Quarter.isin(["2025Q01", "2026Q01"]) &
    ls_original.Economy.isin(CODIGOS_PAISES_UNCTAD) &
    ls_original.Partner.isin(CODIGOS_PAISES_UNCTAD) &
    (ls_original.Economy < ls_original.Partner),
    ["Quarter", "Economy", "Partner", "Index"]
].dropna(subset=["Index"]).copy()
sel["pais_origen"] = sel.Economy.map(CODIGOS_PAISES_UNCTAD)
sel["pais_destino"] = sel.Partner.map(CODIGOS_PAISES_UNCTAD)
sel = sel.rename(columns={"Quarter": "periodo", "Index": "lsbci"})
lsbci = sel[["periodo", "pais_origen", "pais_destino", "lsbci"]].sort_values(
    ["periodo", "pais_origen", "pais_destino"]).reset_index(drop=True)
assert len(lsbci) == 12 and lsbci.periodo.nunique() == 2
lsbci.to_csv(SUBCARPETAS["datos"] / "LSBCI_BILATERAL_4_PAISES_DERIVADO.csv", index=False)

coordenadas = pd.DataFrame(GEOREFERENCIACION,
    columns=['codigo_puerto','latitud','longitud','fuente_url'])
coordenadas['pais']=coordenadas.codigo_puerto.str[:2].map(
    {'PE':'PERU','CL':'CHILE','CO':'COLOMBIA','EC':'ECUADOR'})
coordenadas.to_csv(SUBCARPETAS['control']/'06_GEOREFERENCIACION.csv',index=False)
assert len(coordenadas) == 14 and set(coordenadas.codigo_puerto) == set(CODIGOS)
orden_espacial = coordenadas.codigo_puerto.tolist()
radlat = np.radians(coordenadas.latitud.to_numpy(float))
radlon = np.radians(coordenadas.longitud.to_numpy(float))
aux_a = np.sin((radlat[:, None] - radlat[None, :]) / 2)**2 + (
        np.cos(radlat[:, None]) * np.cos(radlat[None, :]) *
        np.sin((radlon[:, None] - radlon[None, :]) / 2)**2)
distancia = 2 * 6371.0088 * np.arcsin(np.sqrt(np.clip(aux_a, 0, 1)))
np.fill_diagonal(distancia, 0.0)
pd.DataFrame(distancia, index=orden_espacial, columns=orden_espacial).to_csv(
    SUBCARPETAS["resultados"] / "05_DISTANCIAS_KM.csv", index_label="codigo")

def matriz_geografica(k=None):
    w = np.zeros((14, 14), dtype=float)
    if k is not None:
        for i in range(14):
            vecinos = np.argsort(np.where(np.arange(14) == i, np.inf, distancia[i]))[:k]
            w[i, vecinos] = 1.0
    else:
        w = 1 / np.where(distancia == 0, np.inf, distancia)
    return w / w.sum(axis=1, keepdims=True)

MATRICES_W = {"geo_knn3": matriz_geografica(3), "geo_knn4": matriz_geografica(4),
             "geo_inv_distancia": matriz_geografica()}
pais_de_puerto = coordenadas.set_index("codigo_puerto").pais.to_dict()
for periodo in ("2025Q01", "2026Q01"):
    pares = {tuple(sorted([r.pais_origen, r.pais_destino])): float(r.lsbci)
             for r in lsbci.loc[lsbci.periodo == periodo].itertuples()}
    W = np.zeros((14, 14), dtype=float)
    for i, puerto_i in enumerate(orden_espacial):
        for j, puerto_j in enumerate(orden_espacial):
            if i != j and pais_de_puerto[puerto_i] != pais_de_puerto[puerto_j]:
                par = tuple(sorted([pais_de_puerto[puerto_i], pais_de_puerto[puerto_j]]))
                W[i, j] = pares[par]
    MATRICES_W["pais_LSBCI_" + periodo] = W / W.sum(axis=1, keepdims=True)

for nombre, W in MATRICES_W.items():
    assert np.allclose(np.diag(W), 0) and np.allclose(W.sum(axis=1), 1)
    pd.DataFrame(W, index=orden_espacial, columns=orden_espacial).to_csv(
        SUBCARPETAS["resultados"] / ("MATRIZ_W_" + nombre + ".csv"), index_label="codigo")
print("Matrices espaciales construidas:", ", ".join(MATRICES_W))
print("LSBCI bilateral reconstruido desde el CSV mundial original:", len(lsbci), "registros")

### 9.1. Distribución de referencia y pruebas trimestrales

Se realizan 9.999 permutaciones de trayectorias portuarias completas para cada matriz y 1.999 permutaciones para los cálculos de cada trimestre. Los valores $p$ bilaterales se obtienen comparando la distancia del estadístico observado respecto de la media de su distribución permutada. La semilla aleatoria y las matrices utilizadas se conservan para facilitar la reproducción.

In [ ]:
residuos_matriz = panel.pivot(index='codigo',columns='periodo',values='residuo_modelo_1').reindex(
    index=orden_espacial,columns=PERIODOS).to_numpy(float)
residuos_matriz=residuos_matriz-residuos_matriz.mean(axis=0,keepdims=True)

def prueba_moran(residuos,W,n_perm,semilla):
    datos=np.asarray(residuos,float)
    if datos.ndim==1: datos=datos[:,None]
    n=len(datos)
    factor=n/W.sum()
    observado=float(factor*np.einsum('it,ij,jt->',datos,W,datos)/np.sum(datos**2))
    rng=np.random.default_rng(semilla)
    simulaciones=np.empty(n_perm,float)
    for base in range(0,n_perm,400):
        n_bloque=min(400,n_perm-base)
        indices=np.array([rng.permutation(n) for _ in range(n_bloque)])
        d=datos[indices]
        productos=np.einsum('bit,ij,bjt->b',d,W,d,optimize=True)
        denominadores=np.einsum('bit,bit->b',d,d)
        simulaciones[base:base+n_bloque]=factor*productos/denominadores
    media=float(simulaciones.mean())
    p=float((1+np.count_nonzero(np.abs(simulaciones-media)>=np.abs(observado-media))) /
            (n_perm+1))
    return observado,media,p

moran_resultados=[];moran_trimestral=[]
for nombre,W in MATRICES_W.items():
    obs,media,pval=prueba_moran(residuos_matriz,W,N_PERMUTACIONES,SEMILLA+len(nombre)+100)
    moran_resultados.append({'matriz':nombre,'I_moran':obs,'centro_permutaciones':media,
                             'p_bilateral':pval,'permutaciones':N_PERMUTACIONES})
    for t,periodo in enumerate(PERIODOS):
        a,m,p=prueba_moran(residuos_matriz[:,t],W,N_PERMUTACIONES_TRIMESTRE,
                           SEMILLA+7*t+len(nombre))
        moran_trimestral.append({'matriz':nombre,'periodo':periodo,'I_moran':a,
                                 'p_bilateral':p,'permutaciones':N_PERMUTACIONES_TRIMESTRE})
moran_resultados=pd.DataFrame(moran_resultados)
moran_trimestral=pd.DataFrame(moran_trimestral)
moran_resultados.to_csv(SUBCARPETAS['resultados']/'06_MORAN_GLOBAL.csv',index=False)
moran_trimestral.to_csv(SUBCARPETAS['resultados']/'07_MORAN_POR_TRIMESTRE.csv',index=False)
display(moran_resultados.round(5))

## 10. Representación gráfica

Se generan cinco figuras: series observadas de conectividad, trayectorias normalizadas al primer trimestre de 2025, pendientes relativas de los catorce puertos, brecha logarítmica entre Chancay y Callao y resultados del diagnóstico espacial. Las figuras se exportan a PNG y PDF vectorial y se muestran en esta sección para comprobar directamente sus escalas y etiquetas.

In [ ]:
plt.rcParams.update({"font.size": 10, "axes.labelsize": 10, "legend.fontsize": 9,
                     "xtick.labelsize": 9, "ytick.labelsize": 9, "font.family": "sans-serif"})

def exportar_figura(figura, nombre):
    figura.tight_layout()
    figura.savefig(SUBCARPETAS["graficos"] / (nombre + ".png"), dpi=240, bbox_inches="tight")
    figura.savefig(SUBCARPETAS["graficos"] / (nombre + ".pdf"), bbox_inches="tight")
    display(figura)
    plt.close(figura)

fig, ax = plt.subplots(figsize=(9, 4.6))
for puerto in ["PECHY", "PECLL", "CLSAI", "CLVAP", "COBUN", "ECGYE"]:
    ax.plot(PERIODOS, matriz_niveles[puerto], marker="o", label=NOMBRE_PUERTO[puerto])
ax.set(ylabel="PLSCI, puntos del índice", xlabel="Trimestre")
ax.grid(alpha=0.15)
ax.legend(ncol=3, fontsize=8, frameon=False)
exportar_figura(fig, "FIGURA_01_SERIES_PLSCI")

fig, ax = plt.subplots(figsize=(8.8, 4.3))
media_geo_4 = np.exp(matriz_logs[CUATRO_REFERENTES].mean(axis=1))
for serie, nombre in [(matriz_niveles.PECHY, "Chancay"), (matriz_niveles.PECLL, "Callao"),
                       (media_geo_4, "Media geométrica de cuatro puertos")]:
    ax.plot(PERIODOS, 100 * serie / serie.iloc[0], marker="o", label=nombre)
ax.set(xlabel="Trimestre", ylabel="Índice normalizado, 2025T1 = 100")
ax.grid(alpha=0.15)
ax.legend(frameon=False)
exportar_figura(fig, "FIGURA_02_TRAYECTORIAS_NORMALIZADAS")

niveles_est = []
for codigo in CODIGOS:
    tendencia = (float(modelo_principal.params["tendencia_" + codigo])
                 if codigo != "PECLL" else 0.)
    niveles_est.append({"puerto": NOMBRE_PUERTO[codigo],
                        "tendencia_relativa_pct": 100 * np.expm1(tendencia)})
tendencias_puertos = pd.DataFrame(niveles_est).sort_values("tendencia_relativa_pct")
tendencias_puertos.to_csv(SUBCARPETAS["resultados"] / "08_TENDENCIAS_POR_PUERTO.csv", index=False)
fig, ax = plt.subplots(figsize=(8.4, 5.5))
ax.barh(tendencias_puertos.puerto, tendencias_puertos.tendencia_relativa_pct)
ax.axvline(0, ls="--", lw=1)
ax.set(xlabel="Cambio relativo frente a Callao (% por trimestre)")
ax.grid(axis="x", alpha=0.14)
exportar_figura(fig, "FIGURA_03_TENDENCIAS_RELATIVAS")

brecha = series_comparativas["Chancay / Callao"]
ajuste_brecha = ModeloMCO(brecha.to_numpy(), agregar_constante(centro_tiempo)).fit()
fig, ax = plt.subplots(figsize=(8.5, 4.4))
ax.plot(PERIODOS, brecha, marker="o", label="Serie observada")
ax.plot(PERIODOS, ajuste_brecha.fittedvalues, "--", label="Tendencia ajustada")
ax.set(xlabel="Trimestre", ylabel="ln(PLSCI Chancay / PLSCI Callao)")
ax.grid(alpha=0.15)
ax.legend(frameon=False)
exportar_figura(fig, "FIGURA_04_RAZON_CHANCAY_CALLO")

fig, ax = plt.subplots(figsize=(8.2, 3.6))
orden_fig = moran_resultados.iloc[:3]
ax.scatter(orden_fig.I_moran, np.arange(len(orden_fig)))
ax.axvline(-1 / 13, ls="--", lw=1, label="Referencia bajo permutación")
ax.set_yticks(np.arange(len(orden_fig)))
ax.set_yticklabels(["Tres vecinos", "Cuatro vecinos", "Inverso de distancia"])
ax.set(xlabel="I de Moran, residuos del Modelo 1")
for i, r in enumerate(orden_fig.itertuples()):
    ax.annotate("p = " + format(r.p_bilateral, ".3f"), (r.I_moran, i),
                xytext=(7, 9), textcoords="offset points")
ax.margins(x=0.45, y=0.5)
ax.grid(axis="x", alpha=0.15)
exportar_figura(fig, "FIGURA_05_MORAN_RESIDUOS")
print("Se exportaron cinco figuras en PNG y PDF vectorial a:", SUBCARPETAS["graficos"])

## 11. Tablas y resultados de estimación

Las estimaciones se sistematizan en archivos CSV y en un libro Excel con métricas de ajuste, coeficientes, errores estándar, contrastes, sensibilidad y diagnóstico espacial. También se archivan los residuos, las matrices de ponderaciones, el diccionario del panel y la documentación estadística complementaria. Cada conjunto conserva su identificación y su fuente de procedencia.

In [ ]:
SALIDA_XLSX = SUBCARPETAS["resultados"] / "CHANCAY_RESULTADOS_REPRODUCIDOS.xlsx"
LISTA_HOJAS = [
    ("PANORAMA", pd.DataFrame([
        {"concepto": "Fuente principal", "valor": FUENTES_URL["unctad_plsci"]},
        {"concepto": "Inicio del período", "valor": PERIODOS[0]},
        {"concepto": "Final del período", "valor": PERIODOS[-1]},
        {"concepto": "Número de puertos", "valor": 14},
        {"concepto": "Observaciones válidas", "valor": 98},
        {"concepto": "Coeficiente Chancay/Callao", "valor": gamma_chancay},
        {"concepto": "Porcentaje trimestral Chancay/Callao", "valor": 100 * np.expm1(gamma_chancay)}])),
    ("DESCARGAS_HTTP", pd.DataFrame(REGISTRO_DESCARGAS)),
    ("CONVERSION_PDF", pd.DataFrame(REGISTRO_EXTRACCION)),
    ("PUERTOS", resumen_puertos),
    ("BASE_FINAL", pd.read_csv(BASE_FINAL_CSV,encoding="utf-8-sig")),
    ("PANEL_98_RESULTADOS", panel),
    ("MODELOS", metricas),
    ("COEFICIENTES", coeficientes),
    ("CONTRASTES", contrastes),
    ("SENSIBILIDAD", sensibilidad),
    ("LSBCI_PAISES", lsbci),
    ("MORAN_GLOBAL", moran_resultados),
    ("MORAN_TRIMESTRE", moran_trimestral),
    ("ECUADOR_PDF_TABLA5", ecuador),
]
with pd.ExcelWriter(SALIDA_XLSX, engine="openpyxl") as escritor:
    for hoja, dataframe in LISTA_HOJAS:
        dataframe.to_excel(escritor, sheet_name=hoja, index=False)

libro = load_workbook(SALIDA_XLSX)
for hoja in libro.worksheets:
    hoja.freeze_panes = "A2"
    hoja.sheet_view.showGridLines = False
    hoja.auto_filter.ref = hoja.dimensions
    for celda in hoja[1]:
        celda.fill = PatternFill(fill_type="solid", fgColor="183A54")
        celda.font = Font(name="Calibri", size=10, bold=True, color="FFFFFF")
        celda.alignment = Alignment(wrap_text=True)
    for col in hoja.columns:
        letra = col[0].column_letter
        largo = max(min(len(str(celda.value or "")), 54) for celda in list(col)[:250])
        hoja.column_dimensions[letra].width = min(max(largo + 3, 13), 46)
libro.save(SALIDA_XLSX)
print("Excel de resultados creado:", SALIDA_XLSX)

## 12. Verificación de integridad y trazabilidad

La comprobación final coteja el número de observaciones y unidades, la integridad de la base CSV utilizada en las estimaciones, los resultados principales y la existencia de las tablas y figuras. Se genera un manifiesto de archivos con sus huellas SHA-256. La documentación de las descargas y conversiones conserva por separado los estados completos, parciales o pendientes, sin atribuir a los modelos econométricos información que no haya sido efectivamente procesada.

In [ ]:
# Guardamos las fuentes y los resultados que permiten repetir la estimación.
resumen_ejecucion={
    'fecha_utc':datetime.now(timezone.utc).isoformat(),
    'fuente_plsci':FUENTES_URL['unctad_plsci'],
    'base_final_csv':str(BASE_FINAL_CSV),
    'sha256_base_final':sha256_archivo(BASE_FINAL_CSV),
    'sha256_csv_unctad_plsci':sha256_archivo(PLSCI_ORIGEN),
    'fuente_lsbci':FUENTES_URL['unctad_lsbci'],
    'sha256_csv_unctad_lsbci':sha256_archivo(LSBCI_ORIGEN),
    'observaciones':int(len(panel)),
    'puertos':int(panel.codigo.nunique()),
    'trimestres':int(panel.periodo.nunique()),
    'gamma_chancay_callao':float(gamma_chancay),
    'tendencia_relativa_pct':float(100*np.expm1(gamma_chancay)),
    'r2_ajustado':float(modelo_principal.rsquared_adj),
    'archivos_descargados':sum(x['estado']!='ERROR' for x in REGISTRO_DESCARGAS),
    'archivos_con_error':sum(x['estado']=='ERROR' for x in REGISTRO_DESCARGAS),
    'diagnosticos_espaciales':len(moran_resultados),
    'estado_chile':resumen_chile,
    'estado_pdf':pd.DataFrame(REGISTRO_EXTRACCION)['estado'].value_counts().to_dict(),
}
assert resumen_ejecucion['observaciones']==98
assert len(moran_resultados)==5
(SUBCARPETAS['auditoria']/'RESUMEN_EJECUCION.json').write_text(
    json.dumps(resumen_ejecucion,ensure_ascii=False,indent=2),encoding='utf8')

manifiesto=[]
for carpeta in SUBCARPETAS.values():
    for archivo in sorted(carpeta.rglob('*')):
        if archivo.is_file() and archivo.name!='MANIFIESTO_ARCHIVOS_GENERADOS.csv':
            manifiesto.append({'ruta':str(archivo.relative_to(CARPETA_TRABAJO)),
                               'bytes':archivo.stat().st_size,
                               'sha256':sha256_archivo(archivo)})
pd.DataFrame(manifiesto).to_csv(
    SUBCARPETAS['auditoria']/'MANIFIESTO_ARCHIVOS_GENERADOS.csv',index=False,encoding='utf-8-sig')

print('Estimaciones reproducidas y archivos verificados a partir de la base final.')
print('Observaciones:',len(panel),'| Puertos:',panel.codigo.nunique())
print('Tendencia Chancay/Callao:',round(100*np.expm1(gamma_chancay),5),'% por trimestre')
print('Archivos generados:',len(manifiesto))
print('Resultados almacenados en:',CARPETA_TRABAJO)

## Fuentes estadísticas y documentales

- **UNCTADstat.** [Port Liner Shipping Connectivity Index (US.PLSCI)](https://unctadstat.unctad.org/datacentre/reportInfo/US.PLSCI) y [Liner Shipping Bilateral Connectivity Index (US.LSBCI)](https://unctadstat.unctad.org/datacentre/reportInfo/US.LSBCI).
- **Autoridad Portuaria Nacional del Perú.** [Estadísticas de tráfico de carga, 2025](https://www.gob.pe/institucion/apn/informes-publicaciones/6573509-estadisticas-apn-2025-trafico-de-carga).
- **Superintendencia de Transporte de Colombia.** [Tráfico portuario marítimo en Colombia](https://www.datos.gov.co/Transporte/Trafico-Portuario-Mar-timo-En-Colombia/5r3g-zv5z).
- **Observatorio Logístico de Chile.** [Contenedores transferidos (P048)](https://observatorio.observatoriologistico.cl/datos/contenedores-transferidos-en-terminales-estatales-mensualizado), [toneladas transferidas (P012)](https://observatorio.observatoriologistico.cl/datos/toneladas-transferidas-por-puertos-estatales-mensualizado) y [documentación de la API](https://filesprod.observatoriologistico.cl/assets/frontend-graficos/api-datastream-docs/datastreams_api_docs.html).
- **Ecuador.** [Boletines estadísticos portuarios](https://www.mit.gob.ec/boletines-estadisticos-portuarios/).

La URL específica de cada descarga y, cuando corresponde, el identificador SHA-256 se conservan en los registros de procedencia generados durante el procesamiento.